# 15 — Extensões do SHAP binário: PermutationExplainer no RF, RF × SVM com o mesmo explicador, valores negativos e teste do intercepto

Este notebook **estende** o notebook 14 sem refazer nada dele. Ele **lê a pasta do SHAP já calculada** (amostras, fundo, valores SHAP) e calcula **só o que é novo**. Nenhum modelo é treinado, e nenhum arquivo do notebook 14 ou das pastas dos modelos é alterado.

## O que o notebook 14 já tinha (reaproveitado, não recalculado)
`TreeExplainer` no RF e `PermutationExplainer` no SVM, nas amostras estratificadas dos dois dias; o **fundo do SVM** (24 linhas, 50% BENIGN e 50% ataque); as linhas explicadas, os ids, as sementes por pedaço e todos os valores SHAP.

## O que esta extensão acrescenta (só o que muda a interpretação do TCC)
| # | Análise | O que responde | Custo |
|---|---|---|---|
| **1** | **`PermutationExplainer` também no RF**, nas **mesmas linhas do SVM**, nos mesmos dias, com o **mesmo fundo** e as **mesmas permutações** (o `TreeExplainer` continua sendo o principal) | Permite comparar RF e SVM com o **mesmo explicador** | minutos |
| **2** | **Tree × Permutation no RF**, nas mesmas linhas | A escolha do explicador muda a interpretação do RF? ρ de Spearman, Kendall, top 10, Jaccard, razão de magnitude e ρ por linha | segundos |
| **3** | **RF × SVM, os dois com Permutation** (SVM: o do notebook 14, sem recálculo); diferenças entre os dias; padrão × ajustado | Quanto da discordância RF × SVM do notebook 14 era **só o explicador**? | segundos |
| **4** | **Valores negativos, contagem direta, sem modelo** (`fwd_header_length` e `min_seg_size_forward`), por tipo de ataque e por dia (no binário, a contagem do teste é feita no servidor) | O problema é do **dado** ou do **modelo**? | minutos |
| **5** | **Teste do mecanismo no SVM:** distância das linhas explicadas aos vetores de suporte e se a pontuação coincide com a do **intercepto** | Linhas com valor extremo ficam sem vizinhos, e o SVM "cai no intercepto" (todas recebem a mesma pontuação)? | segundos a minutos |

## O que ficou de fora (e por quê)
- **Estabilidade do fundo do SVM** (7 → 14 → 21 linhas): custa ~2 h no multiclasse e é uma checagem de robustez, não um resultado do TCC. O fundo usado é o do notebook 14, **reproduzido e conferido** a partir do refit.
- **`TreeExplainer` interventional** (`TREE_INTERVENTIONAL = True` liga): só serve para separar "algoritmo" de "referência" na análise 2, se os dois explicadores discordarem.

## Cuidados importantes
- **Tree × Permutation mistura duas causas:** o algoritmo e a referência (o `TreeExplainer` usa a distribuição guardada nas árvores; o `PermutationExplainer`, o fundo). Concordância é informativa; discordância **não diz qual causa domina**.
- **A contagem de valores negativos mostra onde e quanto, mas não por quê.** Ela não distingue um erro de coleta de uma característica do tráfego, e o notebook diz isso.
- **O teste dos vetores de suporte vale para o SVM.** O RF trata o valor extremo como "o menor possível" em cada corte, e isso não é testado aqui.
- **O SHAP explica o modelo, não a realidade.** Os valores do SVM são estimativas ruidosas; a importância média de muitas linhas é estável.

## Garantias (verificadas por código)
O fundo do notebook 14 é **reproduzido exatamente** a partir do refit; **nenhuma linha do fundo está entre as explicadas**; as linhas, os pedaços e as **sementes** são os do notebook 14 (o notebook **recalcula 2 linhas do SVM e confere que reproduzem o notebook 14**); aditividade (linha de base + soma dos valores SHAP = saída do modelo); a padronização do SVM é conferida no pipeline; e **nada do notebook 14 nem dos modelos é alterado**.

## Paralelismo, progresso e retomada
Como no notebook 14: tarefas independentes em processos à parte, **vários ao mesmo tempo só se houver memória livre**, barra de progresso, **estimativa medida antes de calcular** (`SO_ESTIMAR = True` só estima e para) e **retomada** (cada tarefa é salva ao terminar).

## Custo esperado
O binário é pequeno: o `PermutationExplainer` do RF leva **minutos**, a contagem de negativos é feita no servidor (só `SELECT`, 2 consultas de agregação), e o teste dos vetores de suporte leva segundos. **O notebook inteiro deve levar de ~10 a 30 min.**

## Antes de rodar
O notebook 14 precisa ter **terminado** (pasta `shap_<data>` com `FINALIZADO`), e **essa pasta não deve ser apagada nem renomeada**. Para a contagem de valores negativos, o notebook consulta o servidor (agregações por tipo de ataque nas tabelas do dia do treino e do teste). O `.env` precisa apontar para o servidor novo. Se você já havia rodado uma versão anterior deste notebook e interrompido, **ela fica numa pasta `extras/x_...` à parte e não atrapalha** (a configuração mudou, então esta versão cria uma pasta nova).


In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

### Ferramentas
O pré-processamento, a leitura em CSV, as métricas, o **escalonador de tarefas condicionado à memória** e as **proteções** (sobrescrita e remoção de arquivos bloqueadas nas pastas dos modelos e na execução do notebook 14).

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# ---------- métricas multiclasse (a partir da matriz de confusão; aceitam pesos por linha) ----------
def metricas_multi(y, proba, w=None):
    y = np.asarray(y, dtype=int); proba = np.asarray(proba, dtype=float); K_ = proba.shape[1]; pred = proba.argmax(axis=1); w_ = None if w is None else np.asarray(w, dtype=float)
    cm = np.zeros((K_, K_)); np.add.at(cm, (y, pred), 1.0 if w_ is None else w_)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)                  # total, linhas por classe verdadeira, previstas por classe, acertos
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k).astype(int)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, proba[:, k], sample_weight=w_)
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t}

# autoteste das ferramentas (não toca no servidor): confere as métricas multiclasse contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc, precision_score as _prec
_r = np.random.default_rng(0); _y = _r.integers(0, 4, 400); _P = _r.dirichlet(np.ones(4), 400); _P[np.arange(400), _y] += 0.35; _P /= _P.sum(axis=1, keepdims=True); _w = _r.uniform(0.5, 3, 400); _pr = _P.argmax(axis=1)
for _ww in (None, _w):
    _e, _a = metricas_multi(_y, _P, _ww)
    assert abs(_e["acuracia"] - _acc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr, sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["f1_macro"] - _f1(_y, _pr, average="macro", sample_weight=_ww)) < 1e-12 and abs(_e["f1_ponderado"] - _f1(_y, _pr, average="weighted", sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["mcc"] - _mcc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["precisao_macro"] - _prec(_y, _pr, average="macro", sample_weight=_ww, zero_division=0)) < 1e-12, _e
    assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P, multi_class="ovr", average="macro", sample_weight=_ww)) < 1e-12, _e
assert abs(metricas_multi(np.array([0, 1, 2]), np.eye(3))[0]["mcc"] - 1.0) < 1e-12
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")


def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas, texto=()):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype={c: (str if c in texto else np.float64) for c in colunas}, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=(object if c in texto else np.float64)) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        msg = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in msg or msg.startswith("Code: 241"):
            raise ErroMemoriaServidor(msg[:600])
        raise Exception(msg[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas, texto=()):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas, texto)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

_b = Barra(2, "autoteste"); _b.avancar(2); assert recall_por_tipo(np.array([0, 1, 1]), np.array([.1, .9, .2]), np.array([0, 1, 1]), 0.5, 2)[1] == 0.5
print("Autoteste das ferramentas do teste OK")

# ---------- métricas multiclasse para arrays grandes (sem cópias em float64; confusão por contagem) ----------
def metricas_multi_grande(y, P):
    y = np.asarray(y, dtype=np.int64); K_ = P.shape[1]; pred = P.argmax(axis=1)
    cm = np.bincount(y * K_ + pred, minlength=K_ * K_).reshape(K_, K_).astype(np.float64)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, np.ascontiguousarray(P[:, k], dtype=np.float64))
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t, "previstas": p}

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; c = (ph + z * z / (2 * n)) / (1 + z * z / n); h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / (1 + z * z / n); return (c - h, c + h)

# autoteste (não toca no servidor): confere contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc
_r = np.random.default_rng(1); _y = _r.integers(0, 4, 500); _P = _r.dirichlet(np.ones(4), 500); _P[np.arange(500), _y] += 0.3; _P = (_P / _P.sum(axis=1, keepdims=True)).astype(np.float32); _pr = _P.argmax(axis=1)
_e, _a = metricas_multi_grande(_y, _P)
assert abs(_e["acuracia"] - _acc(_y, _pr)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr)) < 1e-12 and abs(_e["f1_macro"] - _f1(_y, _pr, average="macro")) < 1e-12 and abs(_e["mcc"] - _mcc(_y, _pr)) < 1e-12, _e
assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P.astype(np.float64), multi_class="ovr", average="macro")) < 1e-9
_lo, _hi = wilson(50, 100); assert 0.40 < _lo < 0.41 and 0.59 < _hi < 0.60
print("Autoteste das ferramentas do teste multiclasse OK")

# ---------- escalonador de tarefas condicionado à memória (o mesmo do 11m, do 12m e do 14) ----------
import collections
try:
    import shap
except ImportError as e:
    raise ImportError("O pacote shap não está instalado. Com a .venv ativa, rode:  pip install shap   (e reinicie o kernel)") from e
from joblib.externals.loky import ProcessPoolExecutor as LokyPool
import concurrent.futures
def ram_livre_gib():
    s = globals().get("RAM_LIVRE_SIMULADA")                                                                     # só para testes
    return float(s() if callable(s) else s) if s is not None else (psutil.virtual_memory().available / 2 ** 30 if psutil else float("inf"))

def processos_seguros(desejado, ram_por_proc_gib):
    # Quantos processos simultâneos cabem AGORA: cada um precisa de ram_por_proc × 1,3, e a reserva nunca é tocada. Sempre devolve pelo menos 1 (nunca trava), mas avisa se nem 1 cabe folgado.
    livre = ram_livre_gib() - RAM_RESERVA_GIB
    if not np.isfinite(livre): return int(desejado)
    return int(max(1, min(desejado, livre // (ram_por_proc_gib * 1.3))))

def rodar_tarefas(pendentes, lancar, coletar, barra, limite_s, k_max, ram_proc, rotulo, parar_apos=None):
    # Escalonador de treinos independentes, UM PROCESSO NOVO por treino. Só inicia mais um treino se houver memória livre para ele (ram_proc × 1,3 + reserva) e se o anterior já ocupou a sua memória;
    # com um treino só rodando, sempre inicia o próximo (nunca trava). Se a memória ficar crítica com vários treinos, o mais novo é interrompido e volta para a fila. Treino acima do limite de tempo é encerrado.
    espera_desde = None; fila, rodando, feitos, ultima, stats = collections.deque(pendentes), [], 0, 0.0, {"max_simultaneos": 0, "interrompidos_por_memoria": 0, "ram_proc_final": ram_proc}
    try:
        while fila or rodando:
            while fila and len(rodando) < k_max:
                livre = ram_livre_gib(); agora = time.perf_counter()
                if rodando and (agora - ultima < INTERVALO_PARTIDAS_S or livre - RAM_RESERVA_GIB < ram_proc * 1.3): break
                if not rodando:
                    if livre < RAM_MINIMA_UM_TREINO_GIB:                                  # com nada rodando, espera a memória liberar (até 10 min); depois, para com uma mensagem clara
                        espera_desde = espera_desde or agora
                        if agora - espera_desde > 600: raise RuntimeError(f"Falta memória livre: um treino precisa de pelo menos {RAM_MINIMA_UM_TREINO_GIB:.1f} GiB livres e há {livre:.1f} GiB. Feche o Meet, o Edge e os outros notebooks (ou defina RAM_MINIMA_UM_TREINO_GIB mais baixo, por sua conta e risco).")
                        barra._extra = f"aguardando memória livre ({livre:.1f} GiB; preciso de pelo menos {RAM_MINIMA_UM_TREINO_GIB:.1f} GiB)..."; barra._mostrar(); time.sleep(2.0); break
                    espera_desde = None
                t = fila.popleft(); ex, fut = lancar(t); rodando.append({"t": t, "ex": ex, "fut": fut, "t0": time.perf_counter(), "k": len(rodando) + 1}); ultima = time.perf_counter(); stats["max_simultaneos"] = max(stats["max_simultaneos"], len(rodando))
            for r in list(rodando):
                dec = time.perf_counter() - r["t0"]
                if r["fut"].done():
                    res = r["fut"].result(); r["ex"].shutdown(wait=True); rodando.remove(r); feitos += 1; ram_proc = max(ram_proc, 1.15 * res[0]["pico_ram_mib"] / 1024); stats["ram_proc_final"] = ram_proc; coletar(r["t"], res, r["k"])
                    if parar_apos and feitos >= parar_apos: raise RuntimeError("parada de teste (PARAR_APOS_FITS)")
                elif dec > limite_s: r["ex"].shutdown(wait=True, kill_workers=True); rodando.remove(r); feitos += 1; coletar(r["t"], None, r["k"])
            if len(rodando) > 1 and ram_livre_gib() < RAM_CRITICA_GIB:
                r = rodando.pop(); r["ex"].shutdown(wait=True, kill_workers=True); fila.appendleft(r["t"]); k_max = max(1, k_max - 1); stats["interrompidos_por_memoria"] += 1
                print(f"  ⚠ memória livre crítica ({ram_livre_gib():.2f} GiB): um treino foi interrompido e volta para a fila; o limite de treinos simultâneos agora é {k_max}")
            barra._extra = f"{rotulo} | {len(rodando)} treino(s) rodando (até {k_max}) | memória livre {ram_livre_gib():.1f} GiB | " + (f"mais antigo há {fmt_tempo(max(time.perf_counter() - r['t0'] for r in rodando))}" if rodando else "")
            barra._mostrar(); time.sleep(0.5)
    except BaseException:
        for r in rodando: r["ex"].shutdown(wait=True, kill_workers=True)
        raise
    return stats

# ---------- proteções: o que este notebook NUNCA faz (sobrescrever ou apagar arquivos das pastas dos modelos) ----------
import warnings, shutil
def silenciar_aviso_sklearn():
    # Com muitos núcleos, o scikit-learn às vezes deixa a lista de filtros de aviso VAZIA (condição de corrida entre threads) e passa a emitir, a cada árvore, um aviso inofensivo
    # sobre `delayed`/`Parallel`. Este desvio descarta SÓ essa mensagem; os outros avisos continuam funcionando. Não muda nenhum resultado.
    warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*", category=UserWarning)
    if not getattr(warnings.warn, "_filtrado_sklearn", False):
        _orig = warnings.warn
        def _warn_filtrado(message, *args, **kwargs):
            if isinstance(message, str) and message.startswith("`sklearn.utils.parallel.delayed` should be used with"): return
            return _orig(message, *args, **kwargs)
        _warn_filtrado._filtrado_sklearn = True; warnings.warn = _warn_filtrado
silenciar_aviso_sklearn()

RAIZES_TRAVADAS, CRIADOS, MANTIDOS = [], [], []
def instalar_travas(pasta, permitir_regravar=()):
    # Bloqueia gravação que SOBRESCREVA arquivo existente dentro das pastas protegidas (as dos modelos), e qualquer remoção de arquivo dentro delas. O treino NÃO é bloqueado (o diagnóstico treina modelos de teste).
    # Pode ser chamada para várias pastas; só arquivos temporários (.tmp) e acréscimos (modo "a") podem ser regravados.
    from sklearn.ensemble import RandomForestClassifier as _RF
    from sklearn.svm import SVC as _SVC
    import numpy as _np, pandas as _pd, joblib as _jl, pathlib as _pl
    RAIZES_TRAVADAS.append(str(Path(pasta).resolve())); perm = set(permitir_regravar)
    if getattr(os.remove, "_travado", False): return
    def dentro(alvo):
        try: s_ = str(Path(alvo).resolve())
        except Exception: return False
        return any(s_.startswith(r) for r in RAIZES_TRAVADAS)
    def checa(alvo, acao, permitir_acrescimo=False):
        if not isinstance(alvo, (str, os.PathLike)) or not dentro(alvo): return
        p = Path(alvo).resolve()
        if p.exists() and p.name not in perm and ".tmp" not in p.name and not permitir_acrescimo:
            raise PermissionError(f"BLOQUEADO: {acao} sobrescreveria '{p.name}', que já existe. Este notebook só cria arquivos novos.")
    def embrulha(obj, nome, pos, acao, kw="", acrescimo_ok=False):
        orig = getattr(obj, nome)
        if getattr(orig, "_travado", False): return
        def f(*a, **k):
            alvo = a[pos] if len(a) > pos else k.get(kw)
            checa(alvo, acao, permitir_acrescimo=(acrescimo_ok and k.get("mode") == "a")); return orig(*a, **k)
        f._travado = True; f.__name__ = getattr(orig, "__name__", nome); setattr(obj, nome, f)
    embrulha(_pd.DataFrame, "to_csv", 1, "gravar CSV", "path_or_buf", acrescimo_ok=True); embrulha(_np, "savez", 0, "gravar NPZ", "file"); embrulha(_np, "savez_compressed", 0, "gravar NPZ", "file"); embrulha(_np, "save", 0, "gravar NPY", "file")
    embrulha(_jl, "dump", 1, "gravar modelo", "filename"); embrulha(_pl.Path, "write_text", 0, "gravar arquivo"); embrulha(_pl.Path, "write_bytes", 0, "gravar arquivo")
    embrulha(os, "replace", 1, "substituir arquivo"); embrulha(os, "rename", 1, "renomear arquivo")
    try:
        from matplotlib.figure import Figure; embrulha(Figure, "savefig", 1, "gravar figura", "fname")
    except ImportError: pass
    def _proibe_remocao(orig):
        def f(*a, **k):
            alvo = a[0] if a else None
            if isinstance(alvo, (str, os.PathLike)) and dentro(alvo): raise PermissionError(f"BLOQUEADO: apagar '{Path(alvo).name}'. Este notebook nunca apaga arquivos.")
            return orig(*a, **k)
        f._travado = True; return f
    if not getattr(os.remove, "_travado", False): os.remove = _proibe_remocao(os.remove)
    if not getattr(_pl.Path.unlink, "_travado", False):
        _unl = _pl.Path.unlink
        def _u(self, *a, **k):
            if dentro(self): raise PermissionError(f"BLOQUEADO: apagar '{self.name}'. Este notebook nunca apaga arquivos.")
            return _unl(self, *a, **k)
        _u._travado = True; _pl.Path.unlink = _u

def gravar(caminho, escrever):
    # Cria o arquivo SÓ se ele ainda NÃO existir (nunca sobrescreve). Registra o que foi criado e o que foi mantido.
    caminho = Path(caminho); rotulo = f"{caminho.parent.name}/{caminho.name}"
    if caminho.exists(): MANTIDOS.append(rotulo); return False
    caminho.parent.mkdir(parents=True, exist_ok=True); escrever(caminho); CRIADOS.append(rotulo); return True
def gravar_csv(df, caminho, **kw): return gravar(caminho, lambda p: df.to_csv(p, **kw))
def gravar_json(obj, caminho): return gravar(caminho, lambda p: salvar_json(obj, p))
def tmp_novo(caminho): return Path(caminho).with_name(Path(caminho).stem + ".tmp" + Path(caminho).suffix)

print("Ferramentas da extensão do SHAP prontas | shap", shap.__version__)


Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


Autoteste das ferramentas do teste OK
Autoteste das ferramentas do teste multiclasse OK
Ferramentas da extensão do SHAP prontas | shap 0.54.1


## 1. A execução do notebook 14 que será estendida, parâmetros e modelos
Localiza a pasta `shap_<data>` finalizada, **herda dela** os modelos, as amostras, o tamanho do fundo, as permutações e a semente (mudá-los quebraria a reutilização), define os parâmetros novos e carrega os modelos finais com as proteções.

In [6]:
# ---------- 1.1 Localização da execução do SHAP (notebook 14) que será ESTENDIDA, e parâmetros ----------
# Esta extensão NÃO recalcula nada que o notebook 14 já calculou: ela lê a pasta dele (amostras, fundo, valores SHAP) e só calcula o que é novo.
REPO = Path(globals().get("REPO_ROOT", Path.cwd())); BASE = REPO / "artefatos_modelos"; PASTA_RAIZ = BASE / "shap_binario"
PAI = globals().get("PASTA_SHAP_PAI")
if PAI is None:
    ps = sorted(p for p in PASTA_RAIZ.glob("shap_*") if (p / "FINALIZADO").exists() and (p / "resultados" / "config_shap_final.json").exists())
    assert ps, f"nenhuma execução FINALIZADA do notebook 14 em {PASTA_RAIZ}: rode antes o notebook 14 (ou defina PASTA_SHAP_PAI)"
    if len(ps) > 1: print("⚠ há mais de uma execução finalizada; usando a mais recente. Para escolher outra, defina PASTA_SHAP_PAI:", [p.name for p in ps])
    PAI = ps[-1]
PAI = Path(PAI); assert (PAI / "FINALIZADO").exists(), f"{PAI.name} não está FINALIZADA"
CFG_PAI = json.loads((PAI / "resultados" / "config_shap_final.json").read_text(encoding="utf-8")); assert CFG_PAI["modo"] == "bin", f"a execução-pai é do modo {CFG_PAI['modo']}, não bin"
# os parâmetros da execução-pai são HERDADOS (mudá-los quebraria a reutilização dos resultados e a comparabilidade)
INCLUIR_PADRAO_RF, INCLUIR_PADRAO_SVM = "rf_padrao" in CFG_PAI["modelos"], "svm_padrao" in CFG_PAI["modelos"]
N_POR_GRUPO_RF, N_POR_GRUPO_SVM = CFG_PAI["n_por_grupo"]["RF"], CFG_PAI["n_por_grupo"]["SVM"]; N_BG_POR_GRUPO_SVM, PERMUTACOES_SVM = CFG_PAI["fundo_svm"]["linhas_por_grupo"], CFG_PAI["fundo_svm"]["permutacoes"]; SEMENTE = CFG_PAI["semente"]
_ch = sorted((PAI / "chunks" / "svm_ajustado" / "teste").glob("c_*.npz")); CHUNK_SVM = int(np.load(_ch[0])["b"] - np.load(_ch[0])["a"]); CHUNK_RF = CHUNK_SVM
# --- parâmetros novos desta extensão ---
ESTAB_BG_PASSOS, REPLICA_FUNDO = (1,), False                                         # (a estabilidade do fundo do SVM foi retirada desta versão: custava ~2 h; o fundo é o do notebook 14)
TREE_INTERVENTIONAL = globals().get("TREE_INTERVENTIONAL", False)                 # TreeExplainer interventional com o mesmo fundo (separa "algoritmo" de "referência"); desligado
NEG_ATRIBUTOS = tuple(globals().get("NEG_ATRIBUTOS", ("fwd_header_length", "min_seg_size_forward")))
INCLUIR_PADRAO_RF_PADRAO, INCLUIR_PADRAO_SVM_PADRAO = INCLUIR_PADRAO_RF, INCLUIR_PADRAO_SVM
N_POR_GRUPO_RF_PADRAO, N_POR_GRUPO_SVM_PADRAO, N_BG_POR_GRUPO_SVM_PADRAO, PERMUTACOES_SVM_PADRAO, CHUNK_RF_PADRAO, CHUNK_SVM_PADRAO = N_POR_GRUPO_RF, N_POR_GRUPO_SVM, N_BG_POR_GRUPO_SVM, PERMUTACOES_SVM, CHUNK_RF, CHUNK_SVM
print(f"Execução-pai (notebook 14): {PAI.name} | modelos: {list(CFG_PAI['modelos'])} | amostras: RF {N_POR_GRUPO_RF}, SVM {N_POR_GRUPO_SVM} por grupo | fundo do SVM: {CFG_PAI['fundo_svm']['linhas']} linhas | semente {SEMENTE} | pedaços de {CHUNK_SVM} linhas")

INCLUIR_PADRAO_RF_PADRAO, INCLUIR_PADRAO_SVM_PADRAO = True, True                  # no binário os modelos são pequenos: os quatro entram por padrão
N_POR_GRUPO_RF_PADRAO, N_POR_GRUPO_SVM_PADRAO = 300, 60                            # linhas por tipo de ataque e por dia
N_BG_POR_GRUPO_SVM_PADRAO, PERMUTACOES_SVM_PADRAO = 1, 1                           # fundo do SVM: 1 linha por tipo; 1 permutação (direta e inversa) por linha (use 2 para menos ruído)
CHUNK_RF_PADRAO, CHUNK_SVM_PADRAO = 1000, 40

# ---------- 1.1 Parâmetros comuns ----------
import joblib, hashlib, collections, math, shutil, zlib
from scipy.stats import rankdata
import shap
REPO = Path(globals().get("REPO_ROOT", Path.cwd())); BASE = REPO / "artefatos_modelos"
SEMENTE = globals().get("SEMENTE", 42)
DIAS = ("treino", "teste"); ROT_DIA = {"treino": "dia do treino", "teste": "dia do teste"}                  # "dia do treino" = linhas que o modelo viu no treino (refit); "dia do teste" = o outro dia
INCLUIR_PADRAO_RF = globals().get("INCLUIR_PADRAO_RF", INCLUIR_PADRAO_RF_PADRAO); INCLUIR_PADRAO_SVM = globals().get("INCLUIR_PADRAO_SVM", INCLUIR_PADRAO_SVM_PADRAO)
N_POR_GRUPO_RF = globals().get("N_POR_GRUPO_RF", N_POR_GRUPO_RF_PADRAO); N_POR_GRUPO_SVM = globals().get("N_POR_GRUPO_SVM", N_POR_GRUPO_SVM_PADRAO)       # linhas explicadas por grupo (classe ou tipo de ataque) e por dia; grupos menores entram INTEIROS
N_BG_POR_GRUPO_SVM = globals().get("N_BG_POR_GRUPO_SVM", N_BG_POR_GRUPO_SVM_PADRAO)       # linhas de referência (fundo) do SVM por grupo: o custo do SVM é proporcional ao tamanho do fundo
PERMUTACOES_SVM = globals().get("PERMUTACOES_SVM", PERMUTACOES_SVM_PADRAO)                # permutações por linha no SVM (1 = direta e inversa; mais = menos ruído, custo proporcional)
CHUNK_RF, CHUNK_SVM = globals().get("CHUNK_RF", CHUNK_RF_PADRAO), globals().get("CHUNK_SVM", CHUNK_SVM_PADRAO)         # linhas por tarefa (cada tarefa é salva ao terminar: retomável)
SO_ESTIMAR = globals().get("SO_ESTIMAR", False)                           # True = mede o tempo por linha, mostra a estimativa e PARA antes de calcular
NOVA_EXECUCAO = globals().get("NOVA_EXECUCAO", False)                     # True = não retomar uma execução interrompida
N_SPLITS_ESTAB, N_BOOT, TOP_K = globals().get("N_SPLITS_ESTAB", 50), globals().get("N_BOOT", 200), globals().get("TOP_K", 10)
# --- paralelismo condicionado à memória (a mesma regra do 11m e do 12m) ---
PROCESSOS_MAX = globals().get("PROCESSOS_MAX", 4); RAM_POR_PROCESSO_GIB = globals().get("RAM_POR_PROCESSO_GIB", 1.0)
RAM_RESERVA_GIB = globals().get("RAM_RESERVA_GIB", 1.0); RAM_MINIMA_UM_TREINO_GIB = globals().get("RAM_MINIMA_UM_TREINO_GIB", 0.5); RAM_CRITICA_GIB = globals().get("RAM_CRITICA_GIB", 0.4)
INTERVALO_PARTIDAS_S = globals().get("INTERVALO_PARTIDAS_S", 10); LIMITE_TAREFA_S = globals().get("LIMITE_TAREFA_S", 6 * 3600); PARAR_APOS_TAREFAS = globals().get("PARAR_APOS_TAREFAS", None)       # só para testes
# regras de suficiência da amostra (fixadas ANTES de olhar os resultados)
REGRAS_ESTAB = {"rho_metade_geral": 0.90, "sobreposicao_top10_geral": 7, "rho_convergencia_50pct": 0.95, "rho_metade_grupo": 0.80}
T_SEC = {}
def foto(raiz, rot): return {(rot + "/" + str(f.relative_to(raiz))): (f.stat().st_size, f.stat().st_mtime_ns) for f in Path(raiz).rglob("*") if f.is_file()}
def ultima(base, padrao, exigir):
    ps = sorted(p for p in (BASE / base).glob(padrao) if all((p / f).exists() for f in exigir)); return ps[-1] if ps else None

MODO, GRUPO_ROTULO = "bin", "tipo de ataque"
EXEMPLOS = globals().get("EXEMPLOS", ["Syn", "UDPLag", "BENIGN"])                    # tipos mostrados nos exemplos locais (uma linha certa e uma errada de cada)
PASTA_RAIZ = BASE / "shap_binario"; PASTA_RAIZ.mkdir(exist_ok=True)
PASTA_RF = Path(globals().get("PASTA_RF") or ultima("rf_binario_busca", "rf_busca_*", ("modelo_final_refit.joblib", "FINALIZADO")) or ".")
PASTA_RF_PAD = Path(globals().get("PASTA_RF_PAD") or ultima("rf_binario", "rf_bin_*", ("modelo_final_refit.joblib",)) or ".")
PASTA_SVM = Path(globals().get("PASTA_SVM") or ultima("svm_binario_busca", "svm_busca_*", ("modelo_final_refit.joblib", "FINALIZADO")) or ".")
assert (PASTA_RF / "modelo_final_refit.joblib").exists() and (PASTA_SVM / "modelo_final_refit.joblib").exists(), "faltam os modelos finais (notebooks 07 e 11 do binário concluídos)"
V_REFIT, SEL_TAB, VIEW_TESTE, TAB_TESTE, AUD_TAB = "prep_vw_bal_bin_refit", "prep_bal_bin_selecao", "prep_vw_rf_bin_teste_sem_sobrep", "prep_final_teste", "prep_views_rf_binario"
faltam = [t for t in (V_REFIT, SEL_TAB, VIEW_TESTE, TAB_TESTE, AUD_TAB) if not existe(t)]; assert not faltam, f"Rode antes os notebooks 02, 03 e 04 do binário: faltam {faltam}"
PASTAS_MODELOS = {"svm": PASTA_SVM, "rf_aj": PASTA_RF, **({"rf_pad": PASTA_RF_PAD} if INCLUIR_PADRAO_RF else {})}
SNAP_ANTES = {n: v for r_, p_ in PASTAS_MODELOS.items() for n, v in foto(p_, r_).items()}
for p_ in PASTAS_MODELOS.values(): instalar_travas(p_)
silenciar_aviso_sklearn()
aud = consultar_num(f"SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM {AUD_TAB} ORDER BY posicao"); versao_views = str(aud["versao"].iloc[0])
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist(); MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views); CM = prep.colunas_modelo; D = len(CM)
assert [c for c in colunas_view(V_REFIT) if c not in ("id_linha", "y")] == CM == [c for c in colunas_view(VIEW_TESTE) if c not in ("id_linha", "y")], "a ordem das colunas das views difere"
VERSOES = {"views": versao_views, "selecao": str(consultar(f"SELECT any(versao) AS v FROM {SEL_TAB}")["v"][0])}
cfg_s = json.loads((PASTA_SVM / "config_treino.json").read_text(encoding="utf-8")) if (PASTA_SVM / "config_treino.json").exists() else {}
LIM_SVM = float(cfg_s.get("limiar_padrao", 0.0))
MODELOS = {}
def registrar(chave, rotulo, familia, arquivo, descricao):
    m = joblib.load(arquivo)
    if familia == "RF": assert list(getattr(m, "colunas_modelo_", CM)) == CM, f"{rotulo}: as colunas do modelo diferem das das views"
    lim = 0.5 if familia == "RF" else LIM_SVM
    MODELOS[chave] = dict(rotulo=rotulo, familia=familia, arquivo=Path(arquivo), modelo=m, rotular=(lambda FX, lim=lim: (FX[:, 0] >= lim).astype(int)), descricao=descricao)
hp_rf = (json.loads((PASTA_RF / "config_treino.json").read_text(encoding="utf-8")).get("hiperparametros", {}) if (PASTA_RF / "config_treino.json").exists() else {}); hp_sv = cfg_s.get("hiperparametros", {})
registrar("rf_ajustado", "RF ajustado", "RF", PASTA_RF / "modelo_final_refit.joblib", f"{hp_rf.get('n_estimators')} árvores | profundidade {hp_rf.get('max_depth')} | folha {hp_rf.get('min_samples_leaf')} | {PASTA_RF.name}")
if INCLUIR_PADRAO_RF: registrar("rf_padrao", "RF padrão", "RF", PASTA_RF_PAD / "modelo_final_refit.joblib", f"100 árvores | profundidade livre | folha 1 | {PASTA_RF_PAD.name}")
registrar("svm_ajustado", "SVM ajustado", "SVM", PASTA_SVM / "modelo_final_refit.joblib", f"C = {hp_sv.get('C')} | gamma = {hp_sv.get('gamma_fator', hp_sv.get('gamma'))}× | {PASTA_SVM.name}")
if INCLUIR_PADRAO_SVM and (PASTA_SVM / "modelo_padrao_refit.joblib").exists(): registrar("svm_padrao", "SVM padrão", "SVM", PASTA_SVM / "modelo_padrao_refit.joblib", f"C = 1 | gamma = 1× | {PASTA_SVM.name}")
elif INCLUIR_PADRAO_SVM: print("⚠ o SVM padrão (modelo_padrao_refit.joblib) não foi encontrado: fica de fora")
print(f"Binário (BENIGN × ataque) | {D} atributos"); print("\nModelos finais (treinados no refit; os mesmos avaliados no teste):")
for k, M in MODELOS.items(): print(f"  {M['rotulo']:13s} limiar natural {0.5 if M['familia'] == 'RF' else LIM_SVM:g} | {M['descricao']}")
print(f"\n🔒 Proteções: sobrescrita e remoção BLOQUEADAS nas pastas dos modelos ({len(SNAP_ANTES)} arquivos já existem e serão conferidos no final); o servidor só é lido.")


Execução-pai (notebook 14): shap_20261109_173230 | modelos: ['rf_ajustado', 'rf_padrao', 'svm_ajustado', 'svm_padrao'] | amostras: RF 300, SVM 60 por grupo | fundo do SVM: 24 linhas | semente 42 | pedaços de 40 linhas
Binário (BENIGN × ataque) | 43 atributos

Modelos finais (treinados no refit; os mesmos avaliados no teste):
  RF ajustado   limiar natural 0.5 | 300 árvores | profundidade 15 | folha 1 | rf_busca_20261024_175105
  RF padrão     limiar natural 0.5 | 100 árvores | profundidade livre | folha 1 | rf_bin_20261021_221311
  SVM ajustado  limiar natural 0 | C = 100.0 | gamma = 4.0× | svm_busca_20261025_004828
  SVM padrão    limiar natural 0 | C = 1 | gamma = 1× | svm_busca_20261025_004828

🔒 Proteções: sobrescrita e remoção BLOQUEADAS nas pastas dos modelos (778 arquivos já existem e serão conferidos no final); o servidor só é lido.


## 2. Dados salvos pelo notebook 14 e verificações
Lê as amostras e o fundo do notebook 14, lê o refit, **reproduz o fundo do notebook 14 e confere que é idêntico** e **garante que o fundo não contém linhas explicadas** (por id e por vetor).

In [7]:
# ---------- amostra estratificada (funções) ----------
def amostra_estratificada(grupo, n_por_grupo, semente):
    # Sorteia até n_por_grupo linhas de CADA grupo (classe ou tipo de ataque). Grupos com menos linhas entram INTEIROS: nenhum grupo minoritário fica de fora.
    # Devolve os índices (em ordem crescente) e a posição de cada linha na ordem aleatória do seu grupo (a "ordem de sorteio"): as primeiras n linhas de cada grupo formam uma subamostra estratificada.
    rng = np.random.default_rng(semente); idx, rk = [], []
    for g in np.unique(grupo):
        pos = np.flatnonzero(grupo == g); sel = rng.permutation(pos)[:n_por_grupo]; idx.append(sel); rk.append(np.arange(len(sel)))
    idx, rk = np.concatenate(idx), np.concatenate(rk); o = np.argsort(idx, kind="stable"); return idx[o], rk[o]
def construir_amk():
    # A mesma amostra principal serve aos dois modelos: o RF usa as primeiras N_POR_GRUPO_RF linhas de cada grupo e o SVM, as primeiras N_POR_GRUPO_SVM (subamostra da do RF).
    out = {}
    for k, M in MODELOS.items():
        n = N_POR_GRUPO_RF if M["familia"] == "RF" else N_POR_GRUPO_SVM; out[k] = {}
        for d in DIAS:
            A = AMOSTRA[d]; m = A["rank"] < n; out[k][d] = {**{c: A[c][m] for c in ("X", "y", "grupo", "id")}, "idx": np.flatnonzero(m)}
    return out
def tab_amostra(dia, n_rf, n_svm):
    A = AMOSTRA[dia]; return pd.DataFrame({"linhas disponíveis": [A["disp"].get(g, 0) for g in GRUPOS_DIA[dia]], "explicadas pelo RF": [int(((A["grupo"] == i) & (A["rank"] < n_rf)).sum()) for i in range(len(GRUPOS_DIA[dia]))],
                                          "explicadas pelo SVM": [int(((A["grupo"] == i) & (A["rank"] < n_svm)).sum()) for i in range(len(GRUPOS_DIA[dia]))]}, index=GRUPOS_DIA[dia])

# ---------- 2.1 Dados salvos pelo notebook 14 e linhas do refit (para os fundos) ----------
t_sec0 = time.perf_counter()
def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
nome_tipo = lambda t: re.sub(r"^DrDoS_", "", str(t))
rf_df = ordenar(ler_view(V_REFIT)); X_rf, y_rf, id_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int), rf_df["id_linha"].astype("uint64").to_numpy(); assert np.isfinite(X_rf).all(), "valores não finitos no refit"
sel = consultar(f"SELECT toString(id_linha) AS id_linha, tipo FROM {SEL_TAB}"); sel["tipo"] = sel["tipo"].map(nome_tipo)
tipo_rf = rf_df["id_linha"].astype("uint64").astype(str).map(dict(zip(sel["id_linha"], sel["tipo"]))); assert tipo_rf.notna().all(), "há linhas do refit sem tipo de ataque"
GRUPOS_TR = ["BENIGN"] + sorted(t for t in set(tipo_rf) if t != "BENIGN"); g_rf = tipo_rf.map({t: i for i, t in enumerate(GRUPOS_TR)}).to_numpy(int); assert ((g_rf == 0) == (y_rf == 0)).all()
INC_SPEC = [(gi, 1) for gi in range(1, len(GRUPOS_TR))] + [(0, len(GRUPOS_TR) - 1)]            # cada passo acrescenta 1 linha por tipo de ataque e o mesmo total de BENIGN (50/50)
def reproduzir_fundo7(idx_tr):
    resto = np.setdiff1d(np.arange(len(y_rf)), idx_tr); g_resto = g_rf[resto]; rng_bg = np.random.default_rng(SEMENTE + 2)
    ib_atq = np.concatenate([rng_bg.permutation(np.flatnonzero(g_resto == gi))[:N_BG_POR_GRUPO_SVM] for gi in range(1, len(GRUPOS_TR))]); ib_ben = rng_bg.permutation(np.flatnonzero(g_resto == 0))[:len(ib_atq)]
    return resto[np.concatenate([ib_ben, ib_atq])]

# ---------- (continuação) amostras, fundos aninhados e verificações ----------
DIAS = ("treino", "teste"); ROT_DIA = {"treino": "dia do treino", "teste": "dia do teste"}
SNAP_PAI = foto(PAI, "pai"); instalar_travas(PAI)                                                  # a pasta do notebook 14 também fica protegida: nada dela é alterado
def chaves(X):
    # Chave EXATA de cada linha: hash de 128 bits dos bytes do vetor (float64); linhas com a mesma chave são idênticas byte a byte.
    X = np.ascontiguousarray(X, dtype=np.float64)
    if X.shape[0] == 0: return []
    mv = memoryview(X).cast("B"); w = X.shape[1] * 8; return [hashlib.blake2b(mv[i * w:(i + 1) * w], digest_size=16).digest() for i in range(X.shape[0])]
AMOSTRA, GRUPOS_DIA = {}, {}
for d in DIAS:
    z = np.load(PAI / "amostras" / f"amostra_{d}.npz", allow_pickle=True); AMOSTRA[d] = {c: z[c] for c in ("X", "y", "grupo", "id", "rank")}; GRUPOS_DIA[d] = [str(g) for g in z["grupos"]]
AMK = construir_amk(); SVM_KEYS = [k for k, M in MODELOS.items() if M["familia"] == "SVM"]; RF_KEYS = [k for k, M in MODELOS.items() if M["familia"] == "RF"]
AMS = AMK[SVM_KEYS[0]]                                                                           # as linhas do SVM (30 por grupo): as MESMAS para todos os modelos nesta extensão
for k in SVM_KEYS: assert all(np.array_equal(AMK[k][d]["id"], AMS[d]["id"]) for d in DIAS)
FUNDO7 = np.load(PAI / "amostras" / "fundo_svm.npy")
# --- o fundo do notebook 14 é reproduzido a partir do refit (prova de que estamos lendo os mesmos dados) ---
idx_tr = np.flatnonzero(np.isin(id_rf, AMOSTRA["treino"]["id"])); assert len(idx_tr) == len(AMOSTRA["treino"]["id"]) and np.array_equal(X_rf[idx_tr], AMOSTRA["treino"]["X"]), "o refit lido agora não coincide com a amostra do dia do treino do notebook 14"
ib7 = reproduzir_fundo7(idx_tr); assert np.array_equal(X_rf[ib7], FUNDO7), "NÃO consegui reproduzir o fundo do notebook 14: confira as sementes e os dados"
print(f"✔ O fundo de {len(FUNDO7)} linhas do notebook 14 foi reproduzido exatamente a partir do refit (mesmas linhas, mesma ordem).")
chaves_expl = set(chaves(AMOSTRA["treino"]["X"])) | set(chaves(AMOSTRA["teste"]["X"]))
BG = {"bg1": FUNDO7}; ID_BG = {"bg1": id_rf[ib7]}; IDX_BG = {"bg1": ib7}; NOME_BG = {"bg1": "fundo base (notebook 14)"}
set_tr = set(idx_tr.tolist()); ch_rf7 = chaves(X_rf[ib7])
# --- garantia: o fundo NÃO contém observações explicadas (por id e por vetor) ---
ids_expl_tr = set(AMOSTRA["treino"]["id"].tolist()); viol_id = {"bg1": len(set(id_rf[ib7].tolist()) & ids_expl_tr)}; viol_vet = {"bg1": int(sum(k_ in chaves_expl for k_ in ch_rf7))}
assert viol_id["bg1"] == 0, f"há linhas do fundo entre as explicadas (por id): {viol_id}"
print("✔ Garantia: nenhuma linha do fundo está entre as linhas explicadas (por id)." + (f" Atenção: {viol_vet['bg1']} linha(s) do fundo do notebook 14 repetem o VETOR de uma linha explicada (ids diferentes; isso é do notebook 14 e não foi alterado)." if viol_vet["bg1"] else " O fundo também não repete nenhum vetor explicado."))
print(f"Fundo (referência) do SVM e do PermutationExplainer do RF: {len(FUNDO7)} linhas do dia do treino, as do notebook 14.")
T_SEC["2.1_carga"] = time.perf_counter() - t_sec0


✔ O fundo de 24 linhas do notebook 14 foi reproduzido exatamente a partir do refit (mesmas linhas, mesma ordem).
✔ Garantia: nenhuma linha do fundo está entre as linhas explicadas (por id). O fundo também não repete nenhum vetor explicado.
Fundo (referência) do SVM e do PermutationExplainer do RF: 24 linhas do dia do treino, as do notebook 14.


## 3. O que será calculado, plano e estimativa
Define o que é novo (as **mesmas linhas, pedaços e sementes** do notebook 14), calcula a impressão digital (para **retomar**), confere a **padronização do SVM** no pipeline, **mede o tempo por linha** e **mostra a estimativa**. Com `SO_ESTIMAR = True` ele para aqui.

In [8]:
# ---------- 3.1 Variantes, plano (pasta, tempos e estimativa) ----------
t_sec0 = time.perf_counter()
def escolher_pedacos(n, frac):
    if frac >= 1: return list(range(n))
    k = max(1, int(round(frac * n))); return sorted(set(np.round(np.linspace(0, n - 1, k)).astype(int).tolist()))
LAYOUT = {d: [(i, min(i + CHUNK_SVM, len(AMS[d]["y"]))) for i in range(0, len(AMS[d]["y"]), CHUNK_SVM)] for d in DIAS}
for d in DIAS:                                                                                    # os pedaços e as sementes são os do notebook 14 (por isso as permutações coincidem)
    for k in SVM_KEYS: assert all((PAI / "chunks" / k / d / f"c_{a:05d}.npz").exists() for a, b in LAYOUT[d]), f"faltam pedaços do notebook 14 ({k}, {d}): a execução-pai está incompleta"
ROWS_ALL = {d: np.arange(len(AMS[d]["y"])) for d in DIAS}
VARIANTES = {"perm_bg1": dict(bg="bg1", pedacos="todos", modelos=list(RF_KEYS), explicador="perm", extra=0, rot="PermutationExplainer no RF, fundo do notebook 14")}      # o SVM já tem: notebook 14
if TREE_INTERVENTIONAL: VARIANTES["tree_interv_bg1"] = dict(bg="bg1", pedacos="todos", modelos=list(RF_KEYS), explicador="tree_interv", extra=0, rot="TreeExplainer interventional no RF, fundo do notebook 14")
def pedacos_var(v, d): return list(range(len(LAYOUT[d])))
def arq_x(k, v, d, a): return PASTA_X / "chunks" / k / v / d / f"c_{a:05d}.npz"
# ---- impressão digital, pasta e retomada ----
IMPRESSAO = hashlib.sha1(json.dumps(_limpar({"pai": PAI.name, "modo": MODO, "variantes": {v: (c["bg"], c["pedacos"], c["modelos"], c["explicador"], c["extra"]) for v, c in VARIANTES.items()}, "semente": SEMENTE, "modelos": {k: MODELOS[k]["arquivo"].stat().st_size for k in MODELOS},
                                           "bg": {n: zlib.crc32(i.astype("uint64").tobytes()) for n, i in ID_BG.items()}}), sort_keys=True).encode()).hexdigest()[:12]
PASTA_X, finalizadas = None, []
for p in sorted((PASTA_RAIZ / "extras").glob("x_*"), reverse=True) if (PASTA_RAIZ / "extras").exists() else []:
    try: c_ = json.loads((p / "config_extras.json").read_text(encoding="utf-8"))
    except (OSError, ValueError): continue
    if (p / "FINALIZADO").exists(): finalizadas.append((p, c_.get("impressao"))); continue
    if not NOVA_EXECUCAO and PASTA_X is None and c_.get("impressao") == IMPRESSAO: PASTA_X = p
if PASTA_X is None and not NOVA_EXECUCAO and any(imp == IMPRESSAO for _, imp in finalizadas):
    raise RuntimeError("Já existe uma execução FINALIZADA desta extensão com a mesma configuração e os mesmos dados: " + ", ".join(p.name for p, imp in finalizadas if imp == IMPRESSAO) + ". Os resultados estão nos arquivos dessa pasta. Para refazer DE PROPÓSITO, defina NOVA_EXECUCAO = True antes desta célula.")
RETOMANDO = PASTA_X is not None
if PASTA_X is None:
    (PASTA_RAIZ / "extras").mkdir(exist_ok=True); RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S"); PASTA_X = PASTA_RAIZ / "extras" / f"x_{RUN_ID}"; PASTA_X.mkdir(parents=True)
    salvar_json({"run_id": RUN_ID, "criado_em": datetime.now().isoformat(timespec="seconds"), "impressao": IMPRESSAO, "pai": PAI.name, "modo": MODO}, PASTA_X / "config_extras.json")
else: RUN_ID = PASTA_X.name.replace("x_", "")
for n, b in BG.items():                                                                           # os fundos (e os ids das linhas) ficam gravados
    gravar(PASTA_X / "fundos" / f"fundo_{n}.npz", lambda p_, n=n, b=b: np.savez_compressed(p_, X=b, id=ID_BG[n], grupo=g_rf[IDX_BG[n]]))
print(f"Pasta da extensão: {PASTA_X}" + ("  (RETOMANDO)" if RETOMANDO else ""))
print(f"Pedaços de {CHUNK_SVM} linhas do notebook 14 (as mesmas linhas, pedaços e sementes): {len(ROWS_ALL['treino'])} linhas do dia do treino e {len(ROWS_ALL['teste'])} do dia do teste, por modelo")
# ---- explicadores (as variantes novas) ----
def _explicar_x(familia, m, X, fundo, perm, modo, semente, explicador):
    D_ = X.shape[1]
    if explicador == "perm":
        if familia == "RF": f = (lambda Z: m.predict_proba(Z)[:, 1:2]) if modo == "bin" else (lambda Z: m.predict_proba(Z))
        else: f = lambda Z: np.asarray(m.decision_function(Z)).reshape(len(Z), -1)
        e = shap.PermutationExplainer(f, fundo, seed=semente)(X, max_evals=2 * D_ * perm + 1); sv = np.asarray(e.values); sv = sv[:, :, None] if sv.ndim == 2 else sv
        fx = f(X); b = np.asarray(e.base_values, float).reshape(len(X), -1); base = b[0]; assert np.allclose(b, b[0], atol=1e-9), "a linha de base varia entre as linhas"
    else:
        ex = shap.TreeExplainer(m, data=fundo, feature_perturbation="interventional"); sv = np.asarray(ex.shap_values(X, check_additivity=False)); fx = m.predict_proba(X); base = np.atleast_1d(np.asarray(ex.expected_value, float))
        if modo == "bin": sv, fx, base = sv[:, :, 1:2], fx[:, 1:2], base[1:2]
    return sv, fx, base
def _worker_x(familia, caminho_modelo, caminho_X, a, b, caminho_fundo, perm, modo, semente, explicador, saida):
    import shap
    m = joblib.load(caminho_modelo); X = np.load(caminho_X, mmap_mode="r")[a:b].copy(); fundo = np.load(caminho_fundo)["X"]
    with Medidor() as med: sv, fx, base = _explicar_x(familia, m, X, fundo, perm, modo, semente, explicador)
    tmp = Path(str(saida) + ".tmp.npz"); np.savez_compressed(tmp, phi=sv.astype(np.float32), fx=fx.astype(np.float32), base=np.asarray(base, float), a=np.int64(a), b=np.int64(b)); os.replace(tmp, saida)
    return {"n": int(b - a), "tempo_s": med.relogio, "cpu_s": med.cpu, "pico_ram_mib": med.pico_mib}, None
# ---- checagens que valem antes de gastar tempo: padronização do SVM no pipeline e o benchmark ----
def checar_padronizacao():
    from sklearn.pipeline import Pipeline
    for k in SVM_KEYS:
        m = MODELOS[k]["modelo"]; assert isinstance(m, Pipeline) and list(m.named_steps) == ["escala", "svm"], f"{MODELOS[k]['rotulo']}: o modelo não é o pipeline (escala + svm) esperado"
        esc = m.named_steps["escala"]; X = AMS["teste"]["X"][:200]; via = np.asarray(m.decision_function(X)); man = np.asarray(m.named_steps["svm"].decision_function(esc.transform(X))); sem = np.asarray(m.named_steps["svm"].decision_function(X))
        assert np.allclose(via, man, atol=1e-9) and not np.allclose(via, sem, atol=1e-3), f"{MODELOS[k]['rotulo']}: a padronização do pipeline não se comporta como esperado"
        print(f"✔ {MODELOS[k]['rotulo']}: o explicador chama o pipeline completo (a padronização é aplicada antes do SVM em toda previsão).")
checar_padronizacao()
print("\nMedindo o tempo por linha (2 linhas, fundo base)...")
chaves_modelos = sorted({k for c in VARIANTES.values() for k in c["modelos"]}); T_LINHA = {}
for k in chaves_modelos:
    M = MODELOS[k]; X2 = AMS["teste"]["X"][:2]; r0 = rss_mib(); t0 = time.perf_counter(); _explicar_x(M["familia"], M["modelo"], X2, BG["bg1"], PERMUTACOES_SVM, MODO, SEMENTE, "perm"); T_LINHA[k] = {"perm": (time.perf_counter() - t0) / 2, "ram": max(rss_mib() - r0, 0.0)}
    if TREE_INTERVENTIONAL and M["familia"] == "RF": t0 = time.perf_counter(); _explicar_x(M["familia"], M["modelo"], X2, BG["bg1"], PERMUTACOES_SVM, MODO, SEMENTE, "tree_interv"); T_LINHA[k]["tree_interv"] = (time.perf_counter() - t0) / 2
    print(f"  {M['rotulo']:13s} {T_LINHA[k]['perm']:7.3f} s por linha (PermutationExplainer, fundo base de {len(BG['bg1'])} linhas)")
PLANO = {}
for v, c in VARIANTES.items():
    for k in c["modelos"]:
        t_lin = (T_LINHA[k]["perm"] * len(BG[c["bg"]]) / len(BG["bg1"])) if c["explicador"] == "perm" else T_LINHA[k]["tree_interv"]
        for d in DIAS:
            for j in pedacos_var(v, d): PLANO[(k, v, d, j)] = (LAYOUT[d][j][1] - LAYOUT[d][j][0]) * t_lin
RAM_ALVO = max(RAM_POR_PROCESSO_GIB, 1.2 * max(T_LINHA[k]["ram"] for k in T_LINHA) / 1024 + 0.3); k_est = processos_seguros(PROCESSOS_MAX, RAM_ALVO); linhas = []
for v, c in VARIANTES.items():
    for k in c["modelos"]:
        tot = sum(t for (kk, vv, d, j), t in PLANO.items() if kk == k and vv == v); linhas.append({"variante": c["rot"], "modelo": MODELOS[k]["rotulo"], "tarefas": sum(len(pedacos_var(v, d)) for d in DIAS), "tempo, 1 processo": fmt_tempo(tot), f"com {k_est} processo(s)": fmt_tempo(tot / k_est)})
print(f"\n== Estimativa (a memória livre agora comporta {k_est} processo(s): {ram_livre_gib():.1f} GiB livres) =="); display(pd.DataFrame(linhas).set_index(["variante", "modelo"]))
print(f"Tempo total estimado: {fmt_tempo(sum(PLANO.values()))} com 1 processo, ~{fmt_tempo(sum(PLANO.values()) / k_est)} com {k_est}")
if SO_ESTIMAR: raise RuntimeError("SO_ESTIMAR = True: parei depois da estimativa. Defina SO_ESTIMAR = False para calcular.")
T_SEC["3.1_plano"] = time.perf_counter() - t_sec0


Pasta da extensão: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\shap_binario\extras\x_20261111_211511
Pedaços de 40 linhas do notebook 14 (as mesmas linhas, pedaços e sementes): 780 linhas do dia do treino e 480 do dia do teste, por modelo
✔ SVM ajustado: o explicador chama o pipeline completo (a padronização é aplicada antes do SVM em toda previsão).
✔ SVM padrão: o explicador chama o pipeline completo (a padronização é aplicada antes do SVM em toda previsão).

Medindo o tempo por linha (2 linhas, fundo base)...
  RF ajustado     0.265 s por linha (PermutationExplainer, fundo base de 24 linhas)
  RF padrão       0.116 s por linha (PermutationExplainer, fundo base de 24 linhas)

== Estimativa (a memória livre agora comporta 1 processo(s): 0.3 GiB livres) ==


tarefas tempo, 1 processo com 1 processo(s)
variante                                         modelo                                                  
PermutationExplainer no RF, fundo do notebook 14 RF ajustado       32          5min 33s          5min 33s
                                                 RF padrão         32          2min 26s          2min 26s

Tempo total estimado: 7min 59s com 1 processo, ~7min 59s com 1


## 4. Cálculo das variantes novas (a parte longa)
Calcula **só o que é novo**, em tarefas independentes (cada uma num processo à parte, vários ao mesmo tempo **só se houver memória livre**), com barra de progresso. Cada tarefa é salva ao terminar. Depois junta tudo e confere a **aditividade**.

In [9]:
# ---------- 4.1 Cálculo das variantes novas (paralelo só com memória; retomável) ----------
t_sec0 = time.perf_counter()
TAREFAS = [(k, v, d, j) for v, c in VARIANTES.items() for k in c["modelos"] for d in DIAS for j in pedacos_var(v, d)]
pend = sorted([t for t in TAREFAS if not arq_x(t[0], t[1], t[2], LAYOUT[t[2]][t[3]][0]).exists()], key=lambda t: -PLANO[t])
n_reuso = sum(len(LAYOUT[d]) for d in DIAS) * len(SVM_KEYS) + sum(len(LAYOUT[d]) for d in DIAS) * len(RF_KEYS)
for t_ in TAREFAS: arq_x(t_[0], t_[1], t_[2], LAYOUT[t_[2]][t_[3]][0]).parent.mkdir(parents=True, exist_ok=True)
print(f"Tarefas novas prontas: {len(TAREFAS) - len(pend)}/{len(TAREFAS)}" + ("  (retomando)" if len(pend) < len(TAREFAS) else "") + f" | a calcular agora: {len(pend)}")
print(f"REAPROVEITADO do notebook 14 (não recalculado): {len(SVM_KEYS)} SVM (fundo base, todas as linhas) e {len(RF_KEYS)} RF (TreeExplainer), nos dois dias.")
STATS_RUN = {}
if pend:
    barra = Barra(int(sum(PLANO[t] for t in pend)) + 1, "Valores SHAP (extensão)"); soma_tarefas, t_sessao, feitas = 0.0, time.perf_counter(), []
    print(f"Memória livre agora: {ram_livre_gib():.1f} GiB → começo com {processos_seguros(PROCESSOS_MAX, RAM_ALVO)} processo(s) (teto {PROCESSOS_MAX}; reavalio a memória a cada partida). A barra é ponderada pelo tempo estimado de cada tarefa.")
    def lancar(t):
        k, v, d, j = t; M = MODELOS[k]; c = VARIANTES[v]; a, b = LAYOUT[d][j]; ex = LokyPool(max_workers=1)
        return ex, ex.submit(_worker_x, M["familia"], str(M["arquivo"]), str(PAI / "amostras" / f"X_SVM_{d}.npy"), a, b, str(PASTA_X / "fundos" / f"fundo_{c['bg']}.npz"), PERMUTACOES_SVM, MODO, SEMENTE + a + c["extra"], c["explicador"], str(arq_x(k, v, d, a)))
    def coletar(t, res, k_ini):
        global soma_tarefas
        assert res is not None, f"a tarefa {t} passou do limite de {fmt_tempo(LIMITE_TAREFA_S)}"
        meta, _ = res; soma_tarefas += meta["tempo_s"]; feitas.append({**meta, "modelo": t[0], "variante": t[1], "dia": t[2], "tarefa": t[3], "simultaneos_na_partida": k_ini})
        barra.avancar(int(PLANO[t]), f"{MODELOS[t[0]]['rotulo']} | {VARIANTES[t[1]]['bg']} | {ROT_DIA[t[2]]} | tarefa {len(feitas)}/{len(pend)} ({meta['n']} linhas em {fmt_tempo(meta['tempo_s'])})")
    try: STATS_RUN = rodar_tarefas(pend, lancar, coletar, barra, LIMITE_TAREFA_S, PROCESSOS_MAX, RAM_ALVO, "SHAP extra", PARAR_APOS_TAREFAS)
    finally:
        rel = time.perf_counter() - t_sessao
        if feitas: pd.DataFrame(feitas).to_csv(PASTA_X / f"tempos_tarefas_{datetime.now():%Y%m%d_%H%M%S}.csv", index=False)
    barra.fechar(); STATS_RUN = {**STATS_RUN, "relogio_s": rel, "soma_tempos_tarefas_s": soma_tarefas, "ganho_paralelismo": soma_tarefas / rel if rel > 0 else float("nan")}
    print(f"✔ {len(feitas)} tarefas em {fmt_tempo(rel)} | no máximo {STATS_RUN['max_simultaneos']} processo(s) simultâneo(s) | ganho observado do paralelismo: {STATS_RUN['ganho_paralelismo']:.2f}× | memória por processo medida: ~{STATS_RUN['ram_proc_final']:.2f} GiB" + (f" | interrompidas por memória: {STATS_RUN['interrompidos_por_memoria']}" if STATS_RUN["interrompidos_por_memoria"] else ""))
# ---- junta os pedaços; carrega o que o notebook 14 já tinha ----
PHIX, FXX, BASEX = {}, {}, {}
for v, c in VARIANTES.items():
    for k in c["modelos"]:
        for d in DIAS:
            zs = [np.load(arq_x(k, v, d, LAYOUT[d][j][0])) for j in pedacos_var(v, d)]; PHIX[(k, v, d)] = np.concatenate([z["phi"] for z in zs]).astype(np.float64); FXX[(k, v, d)] = np.concatenate([z["fx"] for z in zs]).astype(np.float64); BASEX[(k, v, d)] = zs[0]["base"]
            assert np.isfinite(PHIX[(k, v, d)]).all() and len(PHIX[(k, v, d)]) == len(ROWS_ALL[d]), f"valores SHAP incompletos ({k}, {v}, {d})"
ADIT_X = {key: float(np.abs(BASEX[key][None, :] + PHIX[key].sum(axis=1) - FXX[key]).max()) for key in PHIX if VARIANTES[key[1]]["explicador"] == "perm"}
PHI_PAI, FX_PAI, MAP_RF = {}, {}, {}
for k in MODELOS:
    for d in DIAS:
        z = np.load(PAI / "resultados" / f"shap_{k}_{d}.npz"); PHI_PAI[(k, d)] = z["phi"].astype(np.float64); FX_PAI[(k, d)] = z["fx"].astype(np.float64)
        if MODELOS[k]["familia"] == "RF":                                                          # as linhas do RF são 150 por grupo: reordena para as 30 por grupo do SVM, pelo id
            pos = {int(i): r for r, i in enumerate(z["id"])}; MAP_RF[(k, d)] = np.array([pos[int(i)] for i in AMS[d]["id"]]); assert len(MAP_RF[(k, d)]) == len(AMS[d]["id"])
def own_phi(P, rows, d):
    # valores SHAP da saída que se quer explicar em cada linha: multiclasse = a classe REAL; binário = a saída "ataque"
    return P[np.arange(len(P)), :, AMS[d]["y"][rows]] if MODO == "multi" else P[:, :, 0]
def phi_var(k, v, d, rows="todos"):
    # v = "tree" (TreeExplainer do notebook 14, RF) | "perm_bg1" (SVM: do notebook 14; RF: calculado aqui) | "tree_interv_bg1" (opcional)
    if v == "tree": return PHI_PAI[(k, d)][MAP_RF[(k, d)]]
    if v == "perm_bg1" and MODELOS[k]["familia"] == "SVM": return PHI_PAI[(k, d)]
    return PHIX[(k, v, d)]
print("Aditividade das variantes novas (linha de base + soma dos valores SHAP = saída do modelo; erro máximo): " + (" | ".join(f"{MODELOS[k]['rotulo']}/{v}/{d[:5]}: {x:.1e}" for (k, v, d), x in list(ADIT_X.items())[:6]) + (" ..." if len(ADIT_X) > 6 else "") if ADIT_X else "(nenhuma)") + f" | pior caso: {max(ADIT_X.values()) if ADIT_X else 0:.1e}")
T_SEC["4.1_shap_extra"] = time.perf_counter() - t_sec0


Tarefas novas prontas: 0/64 | a calcular agora: 64
REAPROVEITADO do notebook 14 (não recalculado): 2 SVM (fundo base, todas as linhas) e 2 RF (TreeExplainer), nos dois dias.


Memória livre agora: 0.4 GiB → começo com 1 processo(s) (teto 4; reavalio a memória a cada partida). A barra é ponderada pelo tempo estimado de cada tarefa.
Valores SHAP (extensão): concluído em 11min 10s
✔ 64 tarefas em 11min 10s | no máximo 1 processo(s) simultâneo(s) | ganho observado do paralelismo: 0.60× | memória por processo medida: ~1.00 GiB
Aditividade das variantes novas (linha de base + soma dos valores SHAP = saída do modelo; erro máximo): RF ajustado/perm_bg1/trein: 3.3e-08 | RF ajustado/perm_bg1/teste: 3.8e-08 | RF padrão/perm_bg1/trein: 3.6e-08 | RF padrão/perm_bg1/teste: 3.0e-08 | pior caso: 3.8e-08


## 5 e 6. Tree × Permutation no RF (análises 1 e 2) e RF × SVM com o mesmo explicador (análise 3)
Primeiro define as funções de comparação (Spearman, Kendall, top 10, Jaccard, variação do mean \|SHAP\|). **5.2** compara os dois explicadores no RF nas mesmas linhas. **6.1** compara RF e SVM, os dois com `PermutationExplainer`, mostra quanto da discordância do notebook 14 vinha do explicador, e compara os dois dias e o padrão com o ajustado.

In [10]:
# ---------- 5.1 Funções de comparação (Spearman, Kendall, top 10, Jaccard, variação do mean |SHAP|) ----------
from scipy.stats import kendalltau
t_sec0 = time.perf_counter()
def spearman(a, b):
    ra, rb = rankdata(a) - rankdata(a).mean(), rankdata(b) - rankdata(b).mean(); den = math.sqrt((ra ** 2).sum() * (rb ** 2).sum()); return float((ra * rb).sum() / den) if den > 0 else float("nan")
def top_set(s, n=TOP_K): return set(np.argsort(-np.nan_to_num(s))[:n].tolist())
def jaccard(a, b): return len(a & b) / len(a | b) if (a | b) else float("nan")
def partilha(imp): return imp / np.nansum(imp, axis=1, keepdims=True)
def imp_grupos(phi, grupo, G):
    imp = np.full((G, D), np.nan)
    for g in range(G):
        m = grupo == g
        if m.any(): imp[g] = np.abs(phi[m]).mean(axis=0)
    return imp
def resumo_imp(P, rows, d):
    # importâncias de UMA explicação: mean|SHAP| e participação por grupo, e as gerais (peso igual para cada grupo)
    phi = own_phi(P, rows, d); g = AMS[d]["grupo"][rows]; G = len(GRUPOS_DIA[d]); imp = imp_grupos(phi, g, G); sh = partilha(imp)
    with warnings.catch_warnings(): warnings.simplefilter("ignore", RuntimeWarning); return {"imp": imp, "sh": sh, "imp_geral": np.nanmean(imp, axis=0), "sh_geral": np.nanmean(sh, axis=0)}
def cmp_vetores(a, b, ia, ib):
    # a, b: participações; ia, ib: mean|SHAP| (para a variação relativa nos atributos que mais pesam em A)
    ta, tb = top_set(a), top_set(b); o = np.array(sorted(ta)); rel = np.abs(ia[o] - ib[o]) / np.maximum((ia[o] + ib[o]) / 2, 1e-12)
    ra, rb = rankdata(-a), rankdata(-b); return {"ρ": spearman(a, b), "τ": float(kendalltau(a, b)[0]), f"top {TOP_K} em comum": len(ta & tb), "Jaccard": jaccard(ta, tb), "variação rel. do mean|SHAP| (mediana, top)": float(np.median(rel)), "variação rel. do mean|SHAP| (máxima, top)": float(rel.max()),
                                                                          "maior deslocamento de posição (top)": int(np.abs(ra[o] - rb[o]).max())}
def cmp_resumos(A, B, por_grupo=True):
    r = {"geral": cmp_vetores(A["sh_geral"], B["sh_geral"], A["imp_geral"], B["imp_geral"])}
    if por_grupo: r["grupos"] = {}
    return r
def cmp_grupos(A, B, d):
    out = {}
    for gi, g in enumerate(GRUPOS_DIA[d]):
        if np.isnan(A["sh"][gi]).any() or np.isnan(B["sh"][gi]).any(): continue
        out[g] = cmp_vetores(A["sh"][gi], B["sh"][gi], A["imp"][gi], B["imp"][gi])
    return out
T_SEC["5.1_funcoes"] = time.perf_counter() - t_sec0

# ---------- 6.1 Tree × Permutation no RF (item 3) ----------
t_sec0 = time.perf_counter()
def rho_linha(P1, P2, rows, d):
    a, b = own_phi(P1, rows, d), own_phi(P2, rows, d); r = [spearman(x, y) for x, y in zip(a, b) if np.ptp(x) > 0 and np.ptp(y) > 0]; return float(np.nanmedian(r)) if r else float("nan")
TVP, TVP_G, IMP_TP = [], [], {}
for k in RF_KEYS:
    for d in DIAS:
        Pt, Pp = phi_var(k, "tree", d, "todos"), phi_var(k, "perm_bg1", d, "todos"); At, Ap = resumo_imp(Pt, ROWS_ALL[d], d), resumo_imp(Pp, ROWS_ALL[d], d); IMP_TP[(k, d)] = (At, Ap)
        r = cmp_vetores(At["sh_geral"], Ap["sh_geral"], At["imp_geral"], Ap["imp_geral"]); TVP.append({"modelo": MODELOS[k]["rotulo"], "dia": ROT_DIA[d], "linhas": len(ROWS_ALL[d]), **r, "razão mean|SHAP| (Permutation ÷ Tree)": float(Ap["imp_geral"].sum() / At["imp_geral"].sum()), "ρ por linha (mediana)": rho_linha(Pt, Pp, ROWS_ALL[d], d)})
        for g, rr in cmp_grupos(At, Ap, d).items(): TVP_G.append({"modelo": MODELOS[k]["rotulo"], "dia": ROT_DIA[d], GRUPO_ROTULO: g, "ρ": rr["ρ"], f"top {TOP_K} em comum": rr[f"top {TOP_K} em comum"], "Jaccard": rr["Jaccard"]})
TAB_TVP, TAB_TVP_G = pd.DataFrame(TVP), pd.DataFrame(TVP_G)
print(f"== RF: TreeExplainer (exato, referência = distribuição de treino das árvores) × PermutationExplainer (fundo de {len(BG['bg1'])} linhas), nas MESMAS linhas ==")
display(TAB_TVP.set_index(["modelo", "dia"]).round(3)); print(f"\n-- ρ por {GRUPO_ROTULO} entre os dois explicadores (RF) --"); display(TAB_TVP_G.pivot_table(index=["modelo", "dia"], columns=GRUPO_ROTULO, values="ρ").round(2))
for k in RF_KEYS:
    for d in DIAS:
        At, Ap = IMP_TP[(k, d)]; ta, tp = np.argsort(-At["sh_geral"])[:TOP_K], np.argsort(-Ap["sh_geral"])[:TOP_K]
        print(f"{MODELOS[k]['rotulo']} | {ROT_DIA[d]} | top {TOP_K}: Tree: " + ", ".join(CM[j] for j in ta[:6]) + " ... | Permutation: " + ", ".join(CM[j] for j in tp[:6]) + f" ... | só no Tree: {sorted(CM[j] for j in set(ta) - set(tp))} | só no Permutation: {sorted(CM[j] for j in set(tp) - set(ta))}")
print("\n(Atenção: as duas diferenças — o algoritmo e a referência — ficam misturadas; se os rankings concordam, o RF é pouco sensível à escolha; se discordam, a causa só se separa com TREE_INTERVENTIONAL = True.)")
T_SEC["6.1_tree_x_perm"] = time.perf_counter() - t_sec0
# ---------- 7.1 RF × SVM com o MESMO explicador (item 4) ----------
t_sec0 = time.perf_counter(); PARES_MOD = [(a, b) for a, b in (("rf_ajustado", "svm_ajustado"), ("rf_padrao", "svm_padrao")) if a in MODELOS and b in MODELOS]; RXS, RXS_G, IMP_P = [], [], {}
for k in MODELOS:
    for d in DIAS: IMP_P[(k, d)] = resumo_imp(phi_var(k, "perm_bg1", d, "todos"), ROWS_ALL[d], d)
for a, b in PARES_MOD:
    for d in DIAS:
        A, B = IMP_P[(a, d)], IMP_P[(b, d)]; r = cmp_vetores(A["sh_geral"], B["sh_geral"], A["imp_geral"], B["imp_geral"]); rt = spearman(IMP_TP[(a, d)][0]["sh_geral"], B["sh_geral"])
        RXS.append({"par": f"{MODELOS[a]['rotulo']} × {MODELOS[b]['rotulo']}", "dia": ROT_DIA[d], "ρ (RF Permutation × SVM Permutation)": r["ρ"], "ρ (RF Tree × SVM Permutation; notebook 14)": rt, "efeito de trocar o explicador do RF (Δρ)": r["ρ"] - rt, "τ": r["τ"], f"top {TOP_K} em comum": r[f"top {TOP_K} em comum"], "Jaccard": r["Jaccard"]})
        for g, rr in cmp_grupos(A, B, d).items(): RXS_G.append({"par": f"{MODELOS[a]['rotulo']} × {MODELOS[b]['rotulo']}", "dia": ROT_DIA[d], GRUPO_ROTULO: g, "ρ": rr["ρ"], "Jaccard": rr["Jaccard"]})
TAB_RXS, TAB_RXS_G = pd.DataFrame(RXS), pd.DataFrame(RXS_G)
print("== RF × SVM, os dois com PermutationExplainer, as MESMAS linhas e o MESMO fundo ==")
if len(TAB_RXS):
    display(TAB_RXS.set_index(["par", "dia"]).round(3)); print(f"\n-- ρ por {GRUPO_ROTULO} (RF Permutation × SVM Permutation) --"); display(TAB_RXS_G.pivot_table(index=["par", "dia"], columns=GRUPO_ROTULO, values="ρ").round(2))
    for a, b in PARES_MOD:
        for d in DIAS:
            ta, tb = top_set(IMP_P[(a, d)]["sh_geral"]), top_set(IMP_P[(b, d)]["sh_geral"]); print(f"{MODELOS[a]['rotulo']} × {MODELOS[b]['rotulo']} | {ROT_DIA[d]} | em comum no top {TOP_K}: {sorted(CM[j] for j in ta & tb)} | só no RF: {sorted(CM[j] for j in ta - tb)} | só no SVM: {sorted(CM[j] for j in tb - ta)}")
# diferenças entre os dias (com o mesmo explicador) e entre padrão e ajustado
DIAS_X, PXA = [], []
for k in MODELOS:
    At, Ae = IMP_P[(k, "treino")], IMP_P[(k, "teste")]; r = {"modelo": MODELOS[k]["rotulo"], "ρ geral (treino × teste)": spearman(At["sh_geral"], Ae["sh_geral"])}
    for gi, g in enumerate(GRUPOS_DIA["treino"]):
        if g in GRUPOS_DIA["teste"]:
            gj = GRUPOS_DIA["teste"].index(g); a_, b_ = At["sh"][gi], Ae["sh"][gj]
            if not (np.isnan(a_).any() or np.isnan(b_).any()): r[f"ρ {g}"] = spearman(a_, b_)
    delta = Ae["sh_geral"] - At["sh_geral"]; mov = np.argsort(-np.abs(delta))[:5]; r["os 5 atributos que mais mudaram de participação (treino → teste)"] = "; ".join(f"{CM[j]} ({100 * delta[j]:+.1f} p.p.)" for j in mov); r["_mov"] = set(mov.tolist()); DIAS_X.append(r)
TAB_DIAS_X = pd.DataFrame(DIAS_X).set_index("modelo"); print("\n== Dia do treino × dia do teste (PermutationExplainer, o mesmo nos dois modelos) =="); display(TAB_DIAS_X.drop(columns=["_mov"]).round(3))
for a, b in PARES_MOD: print(f"Jaccard entre os 5 atributos que mais mudam entre os dias, {MODELOS[a]['rotulo']} × {MODELOS[b]['rotulo']}: {jaccard(TAB_DIAS_X.loc[MODELOS[a]['rotulo'], '_mov'], TAB_DIAS_X.loc[MODELOS[b]['rotulo'], '_mov']):.2f}")
for fam, (ka, kp) in {"RF": ("rf_ajustado", "rf_padrao"), "SVM": ("svm_ajustado", "svm_padrao")}.items():
    if ka in MODELOS and kp in MODELOS:
        for d in DIAS:
            A, B = IMP_P[(ka, d)], IMP_P[(kp, d)]; r = cmp_vetores(A["sh_geral"], B["sh_geral"], A["imp_geral"], B["imp_geral"]); PXA.append({"família": fam, "dia": ROT_DIA[d], "ρ (ajustado × padrão)": r["ρ"], "τ": r["τ"], f"top {TOP_K} em comum": r[f"top {TOP_K} em comum"], "Jaccard": r["Jaccard"]})
TAB_PXA = pd.DataFrame(PXA)
if len(TAB_PXA): print("\n== Padrão × ajustado, dentro de cada família (PermutationExplainer) =="); display(TAB_PXA.set_index(["família", "dia"]).round(3))
T_SEC["7.1_rf_x_svm"] = time.perf_counter() - t_sec0


== RF: TreeExplainer (exato, referência = distribuição de treino das árvores) × PermutationExplainer (fundo de 24 linhas), nas MESMAS linhas ==


linhas      ρ      τ  top 10 em comum  Jaccard  variação rel. do mean|SHAP| (mediana, top)  variação rel. do mean|SHAP| (máxima, top)  maior deslocamento de posição (top)  \
modelo      dia                                                                                                                                                                                         
RF ajustado dia do treino     780  0.890  0.734                7    0.538                                       0.508                                      0.725                                   10   
            dia do teste      480  0.899  0.734                7    0.538                                       0.529                                      0.806                                   14   
RF padrão   dia do treino     780  0.879  0.712                7    0.538                                       0.529                                      0.760                                   13   
            dia do teste      480  0.876  0.705                7    0.538                                       0.549                                      0.856                                   16   

                           razão mean|SHAP| (Permutation ÷ Tree)  ρ por linha (mediana)  
modelo      dia                                                                          
RF ajustado dia do treino                                  0.980                  0.840  
            dia do teste                                   0.959                  0.838  
RF padrão   dia do treino                                  0.998                  0.793  
            dia do teste                                   0.977                  0.778


-- ρ por tipo de ataque entre os dois explicadores (RF) --


tipo de ataque             BENIGN   DNS  LDAP  MSSQL   NTP  NetBIOS  Portmap  SNMP  SSDP   Syn  TFTP   UDP  UDPLag  WebDDoS
modelo      dia                                                                                                            
RF ajustado dia do teste     0.93   NaN  0.89   0.88   NaN     0.85     0.88   NaN   NaN  0.92   NaN  0.90    0.90      NaN
            dia do treino    0.92  0.88  0.87   0.86  0.92     0.89      NaN  0.86  0.89  0.78  0.89  0.90    0.78     0.92
RF padrão   dia do teste     0.89   NaN  0.86   0.86   NaN     0.86     0.87   NaN   NaN  0.87   NaN  0.89    0.87      NaN
            dia do treino    0.88  0.87  0.85   0.84  0.91     0.88      NaN  0.83  0.89  0.83  0.91  0.89    0.84     0.91

RF ajustado | dia do treino | top 10: Tree: avg_fwd_segment_size, bwd_header_length, bwd_iat_max, protocol, bwd_iat_mean, bwd_iat_total ... | Permutation: protocol, init_win_bytes_forward, avg_fwd_segment_size, urg_flag_count, bwd_header_length, fwd_packet_length_max ... | só no Tree: ['bwd_iat_mean', 'bwd_iat_total', 'total_length_of_fwd_packets'] | só no Permutation: ['ack_flag_count', 'max_packet_length', 'min_seg_size_forward']
RF ajustado | dia do teste | top 10: Tree: avg_fwd_segment_size, bwd_header_length, protocol, bwd_iat_max, bwd_iat_mean, urg_flag_count ... | Permutation: protocol, init_win_bytes_forward, avg_fwd_segment_size, urg_flag_count, ack_flag_count, fwd_packet_length_max ... | só no Tree: ['bwd_iat_mean', 'bwd_iat_total', 'total_length_of_fwd_packets'] | só no Permutation: ['ack_flag_count', 'fwd_packets_s', 'max_packet_length']
RF padrão | dia do treino | top 10: Tree: avg_fwd_segment_size, bwd_header_length, fwd_packet_length_max, bwd_iat_max, bwd_iat_mean, total

ρ (RF Permutation × SVM Permutation)  ρ (RF Tree × SVM Permutation; notebook 14)  efeito de trocar o explicador do RF (Δρ)      τ  top 10 em comum  Jaccard
par                        dia                                                                                                                                                                       
RF ajustado × SVM ajustado dia do treino                                 0.293                                       0.206                                     0.087  0.207                4    0.250
                           dia do teste                                  0.425                                       0.310                                     0.115  0.296                6    0.429
RF padrão × SVM padrão     dia do treino                                 0.351                                       0.201                                     0.150  0.238                6    0.429
                           dia do teste                                  0.354                                       0.290                                     0.064  0.254                5    0.333

... (2 outputs left out)

RF ajustado × SVM ajustado | dia do treino | em comum no top 10: ['avg_fwd_segment_size', 'fwd_packet_length_max', 'init_win_bytes_forward', 'protocol'] | só no RF: ['ack_flag_count', 'bwd_header_length', 'bwd_iat_max', 'max_packet_length', 'min_seg_size_forward', 'urg_flag_count'] | só no SVM: ['act_data_pkt_fwd', 'bwd_packet_length_min', 'flow_bytes_s', 'flow_iat_max', 'flow_iat_mean', 'flow_iat_std']
RF ajustado × SVM ajustado | dia do teste | em comum no top 10: ['ack_flag_count', 'avg_fwd_segment_size', 'fwd_packet_length_max', 'fwd_packets_s', 'init_win_bytes_forward', 'protocol'] | só no RF: ['bwd_header_length', 'bwd_iat_max', 'max_packet_length', 'urg_flag_count'] | só no SVM: ['bwd_packet_length_min', 'down_up_ratio', 'flow_bytes_s', 'min_seg_size_forward']
RF padrão × SVM padrão | dia do treino | em comum no top 10: ['ack_flag_count', 'avg_bwd_segment_size', 'avg_fwd_segment_size', 'fwd_packet_length_max', 'protocol', 'urg_flag_count'] | só no RF: ['bwd_header_length', 'init

,ρ geral (treino × teste),ρ BENIGN,ρ LDAP,ρ MSSQL,ρ NetBIOS,ρ Syn,ρ UDP,ρ UDPLag,os 5 atributos que mais mudaram de participação (treino → teste)
modelo,,,,,,,,,
RF ajustado,0.979,0.954,0.961,0.958,0.963,0.866,0.991,0.818,protocol (+1.2 p.p.); min_seg_size_forward (-0.9 p.p.); fwd_packets_s (+0.8 p.p.); avg_fwd_segment_size (+0.7 p.p.);...
RF padrão,0.962,0.938,0.936,0.949,0.970,0.831,0.984,0.825,protocol (+1.1 p.p.); min_seg_size_forward (-1.0 p.p.); avg_fwd_segment_size (+0.9 p.p.); ack_flag_count (+0.7 p.p.)...
SVM ajustado,0.931,0.979,0.967,0.952,0.878,0.916,0.994,0.925,min_seg_size_forward (+4.0 p.p.); ack_flag_count (+2.8 p.p.); act_data_pkt_fwd (-2.7 p.p.); fwd_packets_s (+2.0 p.p....
SVM padrão,0.953,0.984,0.960,0.950,0.899,0.933,0.998,0.910,min_seg_size_forward (+5.2 p.p.); protocol (-2.3 p.p.); fwd_packets_s (+1.3 p.p.); bwd_packet_length_min (-1.2 p.p.)...


Jaccard entre os 5 atributos que mais mudam entre os dias, RF ajustado × SVM ajustado: 0.43
Jaccard entre os 5 atributos que mais mudam entre os dias, RF padrão × SVM padrão: 0.43

== Padrão × ajustado, dentro de cada família (PermutationExplainer) ==


ρ (ajustado × padrão)      τ  top 10 em comum  Jaccard
família dia                                                                  
RF      dia do treino                  0.955  0.841                8    0.667
        dia do teste                   0.934  0.805                7    0.538
SVM     dia do treino                  0.714  0.526                4    0.250
        dia do teste                   0.773  0.590                7    0.538

## 7. Valores negativos: contagem direta (análise 4)
Conta, **sem usar modelo**, a fração de linhas com valor negativo em cada atributo escolhido (e em todos os atributos), por tipo de ataque e por dia. (No binário, a contagem do teste é feita no servidor.)

In [11]:
# ---------- 8.1 Valores negativos: contagem direta, SEM modelo (consultas de agregação no servidor, só SELECT; e as linhas do refit já lidas) ----------
t_sec0 = time.perf_counter()
NJ = [CM.index(a) for a in NEG_ATRIBUTOS if a in CM]; assert NJ, f"nenhum dos atributos {NEG_ATRIBUTOS} está entre os atributos do modelo"
TAB_TREINO_ALL, TAB_TESTE_ALL = "prep_final_treino", "prep_final_teste"
cols_q = ", ".join(f"countIf({q(c)} < 0) AS {q('neg_' + c)}" for c in prep.features); LIN, TODOS = [], []
def agrega(tab, extra_where, dia, nivel):
    df = consultar_num(f"SELECT label_multi AS tipo, count() AS n, {cols_q} FROM {tab} {extra_where} GROUP BY label_multi"); df["tipo"] = df["tipo"].map(nome_tipo); df = df.groupby("tipo", as_index=False).sum(numeric_only=True)
    for r in df.itertuples(index=False):
        rd = r._asdict()
        for c in prep.features: TODOS.append({"dia": dia, GRUPO_ROTULO: rd["tipo"], "atributo": c, "fração negativa": rd["neg_" + c] / rd["n"]})
        for j in NJ:
            if CM[j] in prep.features: LIN.append({"dia": dia, "nível": nivel, GRUPO_ROTULO: rd["tipo"], "atributo": CM[j], "linhas": int(rd["n"]), "linhas com valor negativo": int(rd["neg_" + CM[j]]), "fração": rd["neg_" + CM[j]] / rd["n"]})
agrega(TAB_TREINO_ALL, "", ROT_DIA["treino"], "todas as linhas do dia do treino"); agrega(TAB_TESTE_ALL, "WHERE sobrepoe_treino = 0", ROT_DIA["teste"], "linhas reais do teste")
for gi, g in enumerate(GRUPOS_TR):                                                                                      # linhas do refit (as que o modelo viu)
    m = g_rf == gi; Xg = X_rf[m]
    for j in NJ: neg = Xg[:, j] < 0; LIN.append({"dia": ROT_DIA["treino"], "nível": "linhas do refit (usadas no treino)", GRUPO_ROTULO: g, "atributo": CM[j], "linhas": int(m.sum()), "linhas com valor negativo": int(neg.sum()), "fração": float(neg.mean())})
TAB_NEG, TAB_NEG_TODOS = pd.DataFrame(LIN), pd.DataFrame(TODOS)
print("== Fração de linhas com valor NEGATIVO nos atributos escolhidos (contagem direta; nenhum modelo é usado) =="); display(TAB_NEG.pivot_table(index=[GRUPO_ROTULO], columns=["atributo", "dia", "nível"], values="fração").round(4))
afetados = TAB_NEG_TODOS[TAB_NEG_TODOS["fração negativa"] >= 0.01].groupby("atributo")["fração negativa"].max().sort_values(ascending=False)
print(f"\nAtributos numéricos com valor negativo em pelo menos 1% das linhas de algum {GRUPO_ROTULO} e dia: " + (", ".join(f"{a} ({100 * v:.0f}%)" for a, v in afetados.items()) if len(afetados) else "nenhum"))
print("(No binário não há vetores distintos do teste em disco: a contagem do teste é sobre as linhas reais, direto no servidor.)")
TAB_ERR = pd.DataFrame()
T_SEC["8.1_negativos"] = time.perf_counter() - t_sec0


  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
== Fração de linhas com valor NEGATIVO nos atributos escolhidos (contagem direta; nenhum modelo é usado) ==


atributo           fwd_header_length                                                                      min_seg_size_forward                                                                    
dia                     dia do teste                      dia do treino                                           dia do teste                      dia do treino                                 
nível          linhas reais do teste linhas do refit (usadas no treino) todas as linhas do dia do treino linhas reais do teste linhas do refit (usadas no treino) todas as linhas do dia do treino
tipo de ataque                                                                                                                                                                                    
BENIGN                        0.0007                             0.0040                           0.0040                0.0008                             0.0041                           0.0041
DNS                              NaN                             0.1599                           0.1593                   NaN                             0.1603                           0.1597
LDAP                          0.9031                             0.0802                           0.0761                0.9031                             0.0806                           0.0763
MSSQL                         0.6504                             0.0638                           0.0657                0.6507                             0.0660                           0.0677
NTP                              NaN                             0.2696                           0.2660                   NaN                             0.2715                           0.2682
NetBIOS                       0.9102                             0.1011                           0.0951                0.9210                             0.1501                           0.1409
Portmap                       0.6606                                NaN                              NaN                0.6606                                NaN                              NaN
SNMP                             NaN                             0.0810                           0.0807                   NaN                             0.0815                           0.0813
SSDP                             NaN                             0.1195                           0.1206                   NaN                             0.1252                           0.1259
Syn                           0.0000                             0.0000                           0.0000                0.0000                             0.0000                           0.0000
TFTP                             NaN                             0.1343                           0.1335                   NaN                             0.3617                           0.3530
UDP                           0.1158                             0.0998                           0.1000                0.1311                             0.1110                           0.1076
UDPLag                        0.0000                             0.0804                           0.0838                0.0000                             0.0846                           0.0894
WebDDoS                          NaN                             0.0000                           0.0000                   NaN                             0.0000                           0.0000


Atributos numéricos com valor negativo em pelo menos 1% das linhas de algum tipo de ataque e dia: init_win_bytes_backward (100%), init_win_bytes_forward (100%), min_seg_size_forward (92%), fwd_header_length (91%)
(No binário não há vetores distintos do teste em disco: a contagem do teste é sobre as linhas reais, direto no servidor.)


## 8. O SVM cai no intercepto? (análise 5)
Para as linhas explicadas, mede a **similaridade máxima a um vetor de suporte** do SVM e se a pontuação coincide com a do **intercepto** (nenhum vetor de suporte contribui), separando as linhas **com** e **sem** valor negativo.

In [12]:
# ---------- 9.1 O SVM "cai no intercepto" quando o vetor está longe de todos os vetores de suporte? (teste do mecanismo, sem treinar nada) ----------
from sklearn.metrics.pairwise import pairwise_distances_argmin_min
t_sec0 = time.perf_counter(); KER = []
# Hipótese: com valores extremos (como -1 bilhão), o vetor fica LONGE de todos os vetores de suporte, a similaridade RBF vai a ~0 e a função de decisão fica só com o intercepto: todas essas linhas recebem a MESMA pontuação.
def sim_e_escore(k, X):
    m = MODELOS[k]["modelo"]; esc, svm = m.named_steps["escala"], m.named_steps["svm"]; Xs = esc.transform(X); _, dmin = pairwise_distances_argmin_min(Xs, svm.support_vectors_)      # distância ao vetor de suporte MAIS PRÓXIMO (espaço padronizado)
    return np.exp(-svm._gamma * dmin ** 2), np.asarray(m.decision_function(X)).reshape(len(X), -1)
ROTULO_SAIDA = lambda S: S.argmax(axis=1) if MODO == "multi" else (S[:, 0] >= 0).astype(int)
for k in SVM_KEYS:
    M = MODELOS[k]; s_int = np.asarray(M["modelo"].decision_function(np.full((1, D), 1e9))).reshape(-1); cls_int = int(ROTULO_SAIDA(s_int[None, :])[0])           # pontuação sem nenhuma contribuição dos vetores de suporte
    print(f"{M['rotulo']}: pontuação só com o intercepto (nenhum vetor de suporte contribui) = {np.round(s_int, 3).tolist()} → classe prevista: " + (CLASSES[cls_int] if MODO == "multi" else ("ataque" if cls_int == 1 else "benigno")))
    for d in DIAS:
        A = AMOSTRA[d]; simmax, S = sim_e_escore(k, A["X"]); sem_viz = simmax < 1e-6; igual_int = np.abs(S - s_int[None, :]).max(axis=1) < 1e-6; pred = ROTULO_SAIDA(S)
        negflag = np.zeros(len(A["X"]), bool)
        for j in [CM.index(a) for a in NEG_ATRIBUTOS if a in CM]: negflag |= A["X"][:, j] < 0
        for gi, g in enumerate(GRUPOS_DIA[d]):
            for nome_, mk in (("com valor negativo", (A["grupo"] == gi) & negflag), ("sem valor negativo", (A["grupo"] == gi) & ~negflag)):
                if mk.sum() == 0: continue
                KER.append({"modelo": M["rotulo"], "dia": ROT_DIA[d], GRUPO_ROTULO: g, "linhas": nome_, "n": int(mk.sum()), "similaridade máxima a um vetor de suporte (mediana)": float(np.median(simmax[mk])), "% sem vizinho (similaridade < 1e-6)": 100 * float(sem_viz[mk].mean()),
                            "% com a pontuação do intercepto": 100 * float(igual_int[mk].mean()), "% previstas como a classe do intercepto": 100 * float((pred[mk] == cls_int).mean())})
TAB_KER = pd.DataFrame(KER)
if len(TAB_KER):
    print(f"\n== Distância das linhas explicadas aos vetores de suporte do SVM (similaridade RBF; 1 = igual a um vetor de suporte, 0 = sem nenhum vizinho) ==\n(\"com valor negativo\" = linhas com valor negativo em {', '.join(a for a in NEG_ATRIBUTOS if a in CM)}; {len(AMOSTRA['teste']['y']):,} linhas por dia)")
    display(TAB_KER.set_index(["modelo", "dia", GRUPO_ROTULO, "linhas"]).round(3))
    res = TAB_KER[TAB_KER["linhas"] == "com valor negativo"].groupby(["modelo", "dia"]).apply(lambda g: pd.Series({"n": int(g["n"].sum()), "% sem vizinho": float(np.average(g["% sem vizinho (similaridade < 1e-6)"], weights=g["n"])), "% com a pontuação do intercepto": float(np.average(g["% com a pontuação do intercepto"], weights=g["n"]))}), include_groups=False)
    print("\n-- Resumo nas linhas COM valor negativo (média ponderada pelas linhas) --"); display(res.round(2))
else: print("(nenhuma linha com ou sem valor negativo para analisar)")
print("\n(Este teste só verifica o mecanismo no SVM. Ele NÃO explica o RF, que trata o valor extremo como \"o menor possível\" em cada corte, nem mostra se o valor é um erro de coleta.)")
T_SEC["9.1_kernel"] = time.perf_counter() - t_sec0


SVM ajustado: pontuação só com o intercepto (nenhum vetor de suporte contribui) = [-0.688] → classe prevista: benigno
SVM padrão: pontuação só com o intercepto (nenhum vetor de suporte contribui) = [-0.73] → classe prevista: benigno

== Distância das linhas explicadas aos vetores de suporte do SVM (similaridade RBF; 1 = igual a um vetor de suporte, 0 = sem nenhum vizinho) ==
("com valor negativo" = linhas com valor negativo em fwd_header_length, min_seg_size_forward; 2,400 linhas por dia)


n  similaridade máxima a um vetor de suporte (mediana)  % sem vizinho (similaridade < 1e-6)  % com a pontuação do intercepto  \
modelo       dia           tipo de ataque linhas                                                                                                                                               
SVM ajustado dia do treino BENIGN         com valor negativo    2                                                0.707                                  0.0                              0.0   
                                          sem valor negativo  298                                                0.888                                  0.0                              0.0   
                           DNS            com valor negativo   48                                                0.909                                  0.0                              0.0   
                                          sem valor negativo  252                                                0.891                                  0.0                              0.0   
                           LDAP           com valor negativo   21                                                0.684                                  0.0                              0.0   
                                          sem valor negativo  279                                                0.677                                  0.0                              0.0   
                           MSSQL          com valor negativo   25                                                0.842                                  0.0                              0.0   
                                          sem valor negativo  275                                                0.928                                  0.0                              0.0   
                           NTP            com valor negativo   80                                                0.843                                  0.0                              0.0   
                                          sem valor negativo  220                                                0.800                                  0.0                              0.0   
                           NetBIOS        com valor negativo   42                                                0.956                                  0.0                              0.0   
                                          sem valor negativo  258                                                0.984                                  0.0                              0.0   
                           SNMP           com valor negativo   28                                                0.815                                  0.0                              0.0   
                                          sem valor negativo  272                                                0.909                                  0.0                              0.0   
                           SSDP           com valor negativo   42                                                0.893                                  0.0                              0.0   
                                          sem valor negativo  258                                                0.953                                  0.0                              0.0   
                           Syn            sem valor negativo  300                                                0.870                                  0.0                              0.0   
                           TFTP           com valor negativo  106                                                0.857                                  0.0                              0.0   
                                          sem valor negativo  194                                                0.861                                  0.0                              0.0   
                           UDP  


-- Resumo nas linhas COM valor negativo (média ponderada pelas linhas) --


n  % sem vizinho  % com a pontuação do intercepto
modelo       dia                                                                 
SVM ajustado dia do teste   996.0            0.0                              0.0
             dia do treino  451.0            0.0                              0.0
SVM padrão   dia do teste   996.0            0.0                              0.0
             dia do treino  451.0            0.0                              0.0


(Este teste só verifica o mecanismo no SVM. Ele NÃO explica o RF, que trata o valor extremo como "o menor possível" em cada corte, nem mostra se o valor é um erro de coleta.)


## 9. Gráficos

In [13]:
# ---------- 10.1 Gráficos ----------
t_sec0 = time.perf_counter(); FIGS = {}
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 5))                                                                # Tree × Permutation no RF
    for a_, d in zip(ax, DIAS):
        for k in RF_KEYS:
            At, Ap = IMP_TP[(k, d)]; a_.scatter(100 * At["sh_geral"], 100 * Ap["sh_geral"], s=14, label=MODELOS[k]["rotulo"])
        mx = 100 * max(max(IMP_TP[(k, d)][0]["sh_geral"].max(), IMP_TP[(k, d)][1]["sh_geral"].max()) for k in RF_KEYS) * 1.05; a_.plot([0, mx], [0, mx], "k--", lw=0.8); a_.set_xlabel("participação no TreeExplainer (%)"); a_.set_ylabel("participação no PermutationExplainer (%)"); a_.set_title(f"RF: Tree × Permutation | {ROT_DIA[d]}"); a_.grid(alpha=.3); a_.legend(fontsize=8)
    fig.tight_layout(); FIGS["rf_tree_x_permutation.png"] = fig
    if PARES_MOD:                                                                                                # RF × SVM, mesmo explicador
        fig, ax = plt.subplots(len(PARES_MOD), 2, figsize=(11, 4.6 * len(PARES_MOD)), squeeze=False)
        for i, (a, b) in enumerate(PARES_MOD):
            for j, d in enumerate(DIAS):
                A, B = IMP_P[(a, d)], IMP_P[(b, d)]; ax[i, j].scatter(100 * A["sh_geral"], 100 * B["sh_geral"], s=14)
                for t_ in np.argsort(-(A["sh_geral"] + B["sh_geral"]))[:5]: ax[i, j].annotate(CM[t_], (100 * A["sh_geral"][t_], 100 * B["sh_geral"][t_]), fontsize=7, xytext=(3, 3), textcoords="offset points")
                mx = 100 * max(A["sh_geral"].max(), B["sh_geral"].max()) * 1.05; ax[i, j].plot([0, mx], [0, mx], "k--", lw=0.8); ax[i, j].set_xlabel(f"{MODELOS[a]['rotulo']} (%)"); ax[i, j].set_ylabel(f"{MODELOS[b]['rotulo']} (%)"); ax[i, j].set_title(f"Permutation nos dois | {ROT_DIA[d]} (ρ = {spearman(A['sh_geral'], B['sh_geral']):.2f})"); ax[i, j].grid(alpha=.3)
        fig.tight_layout(); FIGS["rf_x_svm_mesmo_explicador.png"] = fig
    for atr in [CM[j] for j in NJ]:                                                                              # fração de valores negativos
        d_ = TAB_NEG[(TAB_NEG["atributo"] == atr) & (TAB_NEG["nível"].isin(["linhas reais do teste", "linhas do refit", "linhas do refit (usadas no treino)"]))]
        if len(d_):
            pv = d_.pivot_table(index=GRUPO_ROTULO, columns="dia", values="fração"); fig, ax = plt.subplots(figsize=(8, 0.45 * len(pv) + 1.8)); y_ = np.arange(len(pv)); h = 0.38
            for i_, dia in enumerate(pv.columns): ax.barh(y_ + (i_ - 0.5) * h, 100 * pv[dia].fillna(0), h, label=dia)
            ax.set_yticks(y_); ax.set_yticklabels(pv.index); ax.invert_yaxis(); ax.set_xlabel(f"% das linhas com {atr} negativo"); ax.set_title(f"Valores negativos em {atr}, por {GRUPO_ROTULO}"); ax.legend(); ax.grid(axis="x", alpha=.3); fig.tight_layout(); FIGS[f"negativos_{atr}.png"] = fig
    print(f"✔ {len(FIGS)} gráficos gerados:", ", ".join(FIGS))
except ImportError: print("(matplotlib não está instalado: gráficos não gerados)")
T_SEC["10.1_graficos"] = time.perf_counter() - t_sec0


✔ 4 gráficos gerados: rf_tree_x_permutation.png, rf_x_svm_mesmo_explicador.png, negativos_fwd_header_length.png, negativos_min_seg_size_forward.png


## 10. Gravação e conferências
Grava as tabelas, os valores das variantes novas e os gráficos (**só arquivos novos**) e confere: o fundo do notebook 14 reproduzido, o fundo sem linhas explicadas, a **reprodutibilidade** (recalcular o SVM com o fundo base reproduz o notebook 14), a aditividade e que **nada do notebook 14 nem dos modelos foi alterado**.

In [14]:
# ---------- 11.1 Gravação ----------
t_sec0 = time.perf_counter(); RES = PASTA_X / "resultados"
gravar_csv(TAB_TVP, RES / "rf_tree_x_permutation.csv", index=False); gravar_csv(TAB_TVP_G, RES / "rf_tree_x_permutation_por_grupo.csv", index=False); gravar_csv(TAB_RXS, RES / "rf_x_svm_mesmo_explicador.csv", index=False); gravar_csv(TAB_RXS_G, RES / "rf_x_svm_mesmo_explicador_por_grupo.csv", index=False)
gravar_csv(TAB_DIAS_X.drop(columns=["_mov"]).reset_index(), RES / "dia_treino_x_teste_permutation.csv", index=False); gravar_csv(TAB_PXA, RES / "padrao_x_ajustado_permutation.csv", index=False)
gravar_csv(TAB_NEG, RES / "valores_negativos_atributos_escolhidos.csv", index=False); gravar_csv(TAB_NEG_TODOS, RES / "valores_negativos_todos_os_atributos.csv", index=False); gravar_csv(TAB_ERR, RES / "valores_negativos_x_erro_teste.csv", index=False); gravar_csv(TAB_KER, RES / "svm_distancia_aos_vetores_de_suporte.csv", index=False)
for (k, v, d), P in PHIX.items(): gravar(RES / f"shap_{k}_{v}_{d}.npz", lambda p_, k=k, v=v, d=d, P=P: np.savez_compressed(p_, phi=P, fx=FXX[(k, v, d)], base=BASEX[(k, v, d)], linhas=ROWS_ALL[d], id=AMS[d]["id"], atributos=np.array(CM)))
for nome, fig in FIGS.items(): gravar(RES / nome, lambda p_, f=fig: f.savefig(p_, dpi=130, bbox_inches="tight"))
hw = {"so": platform.platform(), "python": sys.version.split()[0], "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
gravar_json({"run_id": RUN_ID, "modo": MODO, "pai": PAI.name, "impressao": IMPRESSAO, "reaproveitado_do_notebook_14": {"svm_fundo_base_todas_as_linhas": SVM_KEYS, "rf_treeexplainer": RF_KEYS}, "variantes": {v: {"fundo": c["bg"], "linhas_do_fundo": int(len(BG[c["bg"]])), "modelos": c["modelos"], "explicador": c["explicador"]} for v, c in VARIANTES.items()}, "fundo_ids": ID_BG["bg1"].tolist(), "semente": SEMENTE, "permutacoes": PERMUTACOES_SVM, "atributos_negativos": [CM[j] for j in NJ], "tree_interventional": TREE_INTERVENTIONAL, "hardware": hw,
             "paralelismo": {"processos_max": PROCESSOS_MAX, "estatisticas_sessao": {k_: (float(v_) if isinstance(v_, (int, float, np.floating)) else v_) for k_, v_ in STATS_RUN.items()}}}, RES / "config_extras_final.json")
T_SEC["11.1_gravacao"] = time.perf_counter() - t_sec0; gravar_csv(pd.DataFrame({"etapa": list(T_SEC), "segundos": [round(v, 1) for v in T_SEC.values()]}), RES / "tempos_etapas.csv", index=False); print(f"✔ Resultados gravados em {RES}")
# ---------- 12.1 Conferências ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("A execução-pai (notebook 14) está FINALIZADA, e o fundo dela foi reproduzido exatamente a partir do refit", (PAI / "FINALIZADO").exists() and np.array_equal(X_rf[ib7], FUNDO7), f"({len(FUNDO7)} linhas)")
verifica("Nenhuma linha do fundo está entre as explicadas (por id)", viol_id["bg1"] == 0, f"({len(FUNDO7)} linhas)")
verifica("As linhas explicadas são as mesmas do notebook 14 (ids e ordem idênticos), para todos os modelos", all(np.array_equal(AMS[d]["id"], np.load(PAI / "resultados" / f"shap_{k}_{d}.npz")["id"]) for k in SVM_KEYS for d in DIAS))
verifica("Aditividade nas variantes novas: linha de base + soma dos valores SHAP = saída do modelo", (max(ADIT_X.values()) if ADIT_X else 0) < 1e-4, f"(erro máximo {max(ADIT_X.values()) if ADIT_X else 0:.1e})")
verifica("Todas as tarefas novas foram concluídas", all(arq_x(t[0], t[1], t[2], LAYOUT[t[2]][t[3]][0]).exists() for t in TAREFAS), f"({len(TAREFAS)} tarefas)")
# reprodutibilidade: recalcula as 2 primeiras linhas do primeiro pedaço com o fundo base e compara com o notebook 14 (mesmas permutações, mesmo fundo → mesmos valores)
k0 = SVM_KEYS[0]; a0, b0 = LAYOUT["teste"][0][0], LAYOUT["teste"][0][0] + 2; sv_, _, _ = _explicar_x("SVM", MODELOS[k0]["modelo"], AMS["teste"]["X"][a0:b0], BG["bg1"], PERMUTACOES_SVM, MODO, SEMENTE + a0, "perm")
verifica("Reprodutibilidade: recalcular o SVM com o fundo base reproduz os valores do notebook 14 (mesmas permutações e mesmo fundo)", np.allclose(sv_, PHI_PAI[(k0, "teste")][a0:b0], atol=1e-6), f"(diferença máxima {np.abs(sv_ - PHI_PAI[(k0, 'teste')][a0:b0]).max():.1e})")
verifica("Frações de valores negativos entre 0 e 1, e as linhas do teste somam o total", bool(TAB_NEG["fração"].between(0, 1).all()) and (TAB_NEG[TAB_NEG["nível"] == "linhas reais do teste"].groupby("atributo")["linhas"].sum().nunique() == 1))
_depois = {**{n: v for r_, p_ in PASTAS_MODELOS.items() for n, v in foto(p_, r_).items()}, **foto(PAI, "pai")}; _mexidos = [n for n, v in {**SNAP_ANTES, **SNAP_PAI}.items() if _depois.get(n) != v]
verifica(f"Nenhum dos {len(SNAP_ANTES) + len(SNAP_PAI)} arquivos que já existiam (modelos e execução do notebook 14) foi alterado, sobrescrito ou apagado", not _mexidos, f"({_mexidos[:3]})" if _mexidos else "")
verifica("Arquivos esperados gravados", all((RES / f).exists() for f in ("rf_tree_x_permutation.csv", "rf_x_svm_mesmo_explicador.csv", "valores_negativos_atributos_escolhidos.csv", "config_extras_final.json", "tempos_etapas.csv")) and len(FIGS) > 0)
assert all(ok), "há checagens reprovadas acima"
(PASTA_X / "FINALIZADO").write_text(datetime.now().isoformat(timespec="seconds"), encoding="utf-8")
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_X}"); print(f"\n⏱ Tempo total do notebook (etapas medidas): {fmt_tempo(float(sum(T_SEC.values())))}")
print(f"\nArquivos NOVOS criados nesta execução ({len(CRIADOS)}): {', '.join(CRIADOS[:12])}{' ...' if len(CRIADOS) > 12 else ''}")
print("\nPróximo passo: me mande o texto desta saída (e, se quiser, os CSV estabilidade_fundo_*.csv, rf_tree_x_permutation*.csv, rf_x_svm_mesmo_explicador*.csv, valores_negativos_*.csv, svm_distancia_aos_vetores_de_suporte.csv e os PNG).")


✔ Resultados gravados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\shap_binario\extras\x_20261111_211511\resultados
✔ A execução-pai (notebook 14) está FINALIZADA, e o fundo dela foi reproduzido exatamente a partir do refit (24 linhas)
✔ Nenhuma linha do fundo está entre as explicadas (por id) (24 linhas)
✔ As linhas explicadas são as mesmas do notebook 14 (ids e ordem idênticos), para todos os modelos 
✔ Aditividade nas variantes novas: linha de base + soma dos valores SHAP = saída do modelo (erro máximo 3.8e-08)
✔ Todas as tarefas novas foram concluídas (64 tarefas)
✔ Reprodutibilidade: recalcular o SVM com o fundo base reproduz os valores do notebook 14 (mesmas permutações e mesmo fundo) (diferença máxima 2.3e-08)
✔ Frações de valores negativos entre 0 e 1, e as linhas do teste somam o total 
✔ Nenhum dos 932 arquivos que já existiam (modelos e execução do notebook 14) foi alterado, sobrescrito ou apagado 
✔ Arquivos esperados gravados 

✔ 9/9 checagens aprova